
#Pose Estimation — Rostro · Manos · Cuerpo

---

## 🎯 Objetivo del notebook

Al terminar este notebook serás capaz de:

| # | Habilidad |
|---|---|
| 1 | Entender qué son los **landmarks** y cómo se organizan en rostro, mano y cuerpo |
| 2 | Usar **MediaPipe** para detección en tiempo real e imágenes estáticas |
| 3 | Extraer **coordenadas x, y, z** de cada punto clave y graficarlas |
| 4 | Calcular **ángulos entre articulaciones** para análisis biomecánico |
| 5 | Detectar **gestos con la mano** (puño, paz, pulgar arriba, etc.) |
| 6 | Detectar **expresiones faciales** (boca abierta, ojos cerrados) |
| 7 | Detectar **caídas y posturas incorrectas** |
| 8 | Usar **YOLOv8-pose** como tecnología alternativa |
| 9 | Comparar las tecnologías: MediaPipe vs OpenPose vs YOLO |
| 10 | Construir una **aplicación completa** con webcam en tiempo real |

---

## 🗺️ Estructura del notebook

```
MÓDULO 0 → Instalación y conceptos base
MÓDULO 1 → Face Landmarks (468 puntos del rostro)
MÓDULO 2 → Hand Landmarks (21 puntos por mano)
MÓDULO 3 → Body Pose (33 puntos del cuerpo)
MÓDULO 4 → Holistic (rostro + manos + cuerpo simultáneo)
MÓDULO 5 → Mediciones y ángulos biomecánicos
MÓDULO 6 → Detección de gestos con la mano
MÓDULO 7 → Detección de caídas y posturas incorrectas
MÓDULO 8 → YOLOv8-pose como alternativa
MÓDULO 9 → Aplicación en tiempo real completa
MÓDULO 10→ Comparativa de tecnologías
```

---

## 📚 Conceptos fundamentales antes de empezar

### ¿Qué es un Landmark?
Un **landmark** (punto clave) es un punto específico en el cuerpo humano que la IA identifica y rastrea.  
Cada landmark tiene **3 coordenadas**:

```
x → posición horizontal  (0.0 = izquierda, 1.0 = derecha de la imagen)
y → posición vertical    (0.0 = arriba,    1.0 = abajo  de la imagen)
z → profundidad          (negativo = más cercano a la cámara)
```

Los valores x, y, z son **normalizados** (entre 0 y 1 aprox) para que funcionen  
independientemente del tamaño de la imagen.

### ¿Qué es la estimación de pose?
Es el proceso de localizar estos landmarks en la imagen usando **redes neuronales**  
entrenadas con millones de imágenes etiquetadas manualmente.

### Pipeline básico de cualquier sistema de pose:
```
IMAGEN/VIDEO
     ↓
DETECCIÓN (¿hay una persona/mano/cara en la imagen?)
     ↓
LOCALIZACIÓN DE LANDMARKS (¿dónde están exactamente los puntos?)
     ↓
POST-PROCESAMIENTO (ángulos, gestos, alertas, visualización)
```

---

> ✅ Ejecuta las celdas en orden con `Shift + Enter`  
> 🔁 Si reinicias el kernel, empieza desde el Módulo 0

---
# 📦 MÓDULO 0 — Instalación y Configuración del Entorno

### Librerías que usaremos

| Librería | Versión | ¿Para qué? |
|---|---|---|
| `mediapipe` | ≥0.10 | Framework de Google para pose en tiempo real |
| `opencv-python` | ≥4.8 | Captura de cámara y dibujo sobre imágenes |
| `numpy` | ≥1.24 | Cálculos matemáticos con matrices de píxeles |
| `matplotlib` | ≥3.7 | Visualización de landmarks y gráficas |
| `Pillow` | ≥10.0 | Manejo de formatos de imagen |
| `pandas` | ≥2.0 | Organización de datos de landmarks en tablas |
| `ultralytics` | ≥8.0 | YOLOv8-pose (módulo 8) |
| `ipywidgets` | ≥8.0 | Controles interactivos en Jupyter |

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 0 — INSTALACIÓN DE LIBRERÍAS                           ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─── VERSIONES FIJADAS — IMPORTANTE ────────────────────────────────────
# mediapipe <= 0.10.21  → última versión que incluye mp.solutions.*
# numpy < 2             → mediapipe 0.10.21 no es compatible con numpy 2.x
# protobuf < 4          → protobuf >= 4 rompe MessageFactory en Python 3.12
# ────────────────────────────────────────────────────────────────────────
%pip install -q "mediapipe==0.10.21" "numpy>=1.24,<2" "protobuf>=3.20,<4"
%pip install -q opencv-python matplotlib Pillow pandas ipywidgets
%pip install -q ultralytics

print("✅ Todas las librerías instaladas.\n")
print("📋 Versiones instaladas:")

import mediapipe as mp
import cv2
import numpy as np
import matplotlib
import pandas as pd
import PIL
import ipywidgets as widgets
import ultralytics
from ultralytics import YOLO

print(f"   MediaPipe   : {mp.__version__}")
print(f"   OpenCV      : {cv2.__version__}")
print(f"   NumPy       : {np.__version__}")
print(f"   Matplotlib  : {matplotlib.__version__}")
print(f"   pandas      : {pd.__version__}")
print(f"   Pillow      : {PIL.__version__}")
print(f"   ipywidgets  : {widgets.__version__}")
print(f"   ultralytics : {ultralytics.__version__}")

print("\n✅ YOLOv8 listo para usarse.")
print("ℹ️ Si alguna librería falla al importar, reinicia el kernel.")

Note: you may need to restart the kernel to use updated packages.


ERROR: Cannot install mediapipe==0.10.21 and protobuf<4 and >=3.20 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✅ Todas las librerías instaladas.

📋 Versiones instaladas:
   MediaPipe   : 0.10.32
   OpenCV      : 4.13.0
   NumPy       : 2.3.5
   Matplotlib  : 3.10.8
   pandas      : 3.0.0
   Pillow      : 12.1.0
   ipywidgets  : 8.1.8
   ultralytics : 8.4.19

✅ YOLOv8 listo para usarse.
ℹ️ Si alguna librería falla al importar, reinicia el kernel.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 0 — IMPORTACIONES GLOBALES, DESCARGA DE MODELOS        ║
# ║             Y FUNCIONES UTILITARIAS                             ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Librerías estándar ────────────────────────────────────────────
import os, io, time, math, threading, warnings, urllib.request
warnings.filterwarnings('ignore')

# ── Visión artificial ─────────────────────────────────────────────
import cv2
import numpy as np
import mediapipe as mp
from PIL import Image

# ── Visualización ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

# ── Datos e interfaz ──────────────────────────────────────────────
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ════════════════════════════════════════════════════════════════════
# NUEVA API: mp.tasks — ¿cómo se organiza?
# ════════════════════════════════════════════════════════════════════
#
# La API mp.solutions (legacy) estaba acoplada al paquete:
#   mp.solutions.pose.Pose(...)
#
# La API mp.tasks (moderna) separa tres conceptos:
#   1. BaseOptions  → indica DÓNDE está el modelo (.task file)
#   2. XxxOptions   → configura hiperparámetros del modelo
#   3. XxxLandmarker → el objeto detector, con método .detect()
#
# Flujo en la nueva API (siempre el mismo patrón):
#
#   opts = vision.FaceLandmarkerOptions(
#       base_options=BaseOptions(model_asset_path='face_landmarker.task'),
#       running_mode=RunningMode.IMAGE,    # IMAGE | VIDEO | LIVE_STREAM
#       ...
#   )
#   with vision.FaceLandmarker.create_from_options(opts) as detector:
#       mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
#       result = detector.detect(mp_img)     # ← detect(), no process()
#       # result.face_landmarks[rostro_idx][landmark_idx].x / .y / .z
#
# DIFERENCIAS IMPORTANTES respecto a la API legacy:
#   Legacy:  detector.process(imagen_rgb)         → result.multi_face_landmarks
#   Nueva:   detector.detect(mp.Image(...))       → result.face_landmarks
#
#   Legacy:  landmark.x, landmark.y               → igual en la nueva API ✅
#   Legacy:  result.multi_face_landmarks[i]       → result.face_landmarks[i]
#   Legacy:  result.multi_hand_landmarks[i]       → result.hand_landmarks[i]
#   Legacy:  result.pose_landmarks.landmark       → result.pose_landmarks[0]
#   Legacy:  result.multi_handedness[i]           → result.handedness[i]

from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core import base_options as base_opts_module
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
from mediapipe.tasks.python.vision import drawing_styles as mp_drawing_styles
from mediapipe.tasks.python.vision import (
    FaceLandmarker, FaceLandmarkerOptions,
    FaceLandmarksConnections,
    HandLandmarker, HandLandmarkerOptions,
    HandLandmarksConnections,
    PoseLandmarker, PoseLandmarkerOptions,
    PoseLandmarksConnections,
    GestureRecognizer, GestureRecognizerOptions,
    RunningMode,
)
BaseOptions = base_opts_module.BaseOptions

# ════════════════════════════════════════════════════════════════════
# DESCARGA DE MODELOS .task
# ════════════════════════════════════════════════════════════════════
#
# La nueva API NO incluye los modelos dentro del paquete pip.
# Hay que descargar archivos .task desde el CDN de Google.
# Cada archivo es un modelo TFLite empacado con metadatos.
#
# Se descargan UNA VEZ y se guardan localmente.
# En Colab, se guardan en /content/ (se pierden al reiniciar).

MODELOS = {
    'face':         ('face_landmarker.task',
                     'https://storage.googleapis.com/mediapipe-models/'
                     'face_landmarker/face_landmarker/float16/1/face_landmarker.task'),
    'hand':         ('hand_landmarker.task',
                     'https://storage.googleapis.com/mediapipe-models/'
                     'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'),
    'pose_lite':    ('pose_landmarker_lite.task',
                     'https://storage.googleapis.com/mediapipe-models/'
                     'pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'),
    'pose_full':    ('pose_landmarker_full.task',
                     'https://storage.googleapis.com/mediapipe-models/'
                     'pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task'),
    'pose_heavy':   ('pose_landmarker_heavy.task',
                     'https://storage.googleapis.com/mediapipe-models/'
                     'pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task'),
    'gesture':      ('gesture_recognizer.task',
                     'https://storage.googleapis.com/mediapipe-models/'
                     'gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'),
}

print("⬇️  Descargando modelos MediaPipe Tasks...")
for nombre, (archivo, url) in MODELOS.items():
    if os.path.exists(archivo) and os.path.getsize(archivo) > 1000:
        print(f"   ✅ {archivo} ya existe ({os.path.getsize(archivo)//1024} KB)")
    else:
        print(f"   ⬇️  Descargando {archivo}...", end=' ', flush=True)
        try:
            urllib.request.urlretrieve(url, archivo)
            kb = os.path.getsize(archivo) // 1024
            print(f"✅ {kb} KB")
        except Exception as e:
            print(f"❌ Error: {e}")

# Rutas cortas para usar en el resto del notebook
MODEL_FACE       = MODELOS['face'][0]
MODEL_HAND       = MODELOS['hand'][0]
MODEL_POSE_LITE  = MODELOS['pose_lite'][0]
MODEL_POSE_FULL  = MODELOS['pose_full'][0]
MODEL_POSE_HEAVY = MODELOS['pose_heavy'][0]
MODEL_GESTURE    = MODELOS['gesture'][0]

# ════════════════════════════════════════════════════════════════════
# FUNCIONES UTILITARIAS (idénticas a la versión legacy)
# ════════════════════════════════════════════════════════════════════

def bgr_a_rgb(imagen_bgr):
    """Convierte imagen BGR (OpenCV) → RGB (MediaPipe/Matplotlib)."""
    return cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

def rgb_a_mp_image(imagen_rgb):
    """
    Convierte un array NumPy RGB a mp.Image, que es el formato
    que aceptan todos los detectores de la nueva API.
    
    Diferencia clave con la API legacy:
      Legacy: detector.process(imagen_rgb)        ← acepta NumPy directamente
      Nueva:  detector.detect(mp.Image(...))      ← requiere mp.Image wrapper
    """
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)

def cargar_imagen(ruta_o_url):
    """Carga imagen desde disco o URL y devuelve array NumPy RGB."""
    if ruta_o_url.startswith('http'):
        import urllib.request
        with urllib.request.urlopen(ruta_o_url) as resp:
            datos = resp.read()
        return np.array(Image.open(io.BytesIO(datos)).convert('RGB'))
    else:
        img_bgr = cv2.imread(ruta_o_url)
        if img_bgr is None:
            raise FileNotFoundError(f"No se encontró: {ruta_o_url}")
        return bgr_a_rgb(img_bgr)

def mostrar_imagen(img_rgb, titulo='Imagen', figsize=(10, 6)):
    """Muestra imagen RGB en el notebook."""
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb); plt.title(titulo, fontsize=13, fontweight='bold')
    plt.axis('off'); plt.tight_layout(); plt.show()

def mostrar_comparacion(img1, img2, titulo1='Original', titulo2='Procesada', figsize=(16, 7)):
    """Muestra dos imágenes lado a lado."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(img1); axes[0].set_title(titulo1, fontsize=12, fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(img2); axes[1].set_title(titulo2, fontsize=12, fontweight='bold'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

def calcular_angulo(punto_a, punto_b, punto_c):
    """
    Ángulo en grados en el vértice B formado por A-B-C.
    Funciona igual con landmarks de la API legacy y la nueva API,
    ya que ambas tienen los mismos atributos .x, .y, .z.
    """
    a = np.array(punto_a[:2]); b = np.array(punto_b[:2]); c = np.array(punto_c[:2])
    ba = a - b; bc = c - b
    mag_ba = np.linalg.norm(ba); mag_bc = np.linalg.norm(bc)
    if mag_ba == 0 or mag_bc == 0: return 0.0
    coseno = np.clip(np.dot(ba, bc) / (mag_ba * mag_bc), -1.0, 1.0)
    return round(math.degrees(math.acos(coseno)), 1)

def landmark_a_pixeles(landmark, alto, ancho):
    """
    Convierte coordenadas normalizadas (0-1) a píxeles absolutos.
    Compatible con landmarks de AMBAS APIs (mismo formato .x, .y).
    """
    return (int(landmark.x * ancho), int(landmark.y * alto))

print("\n✅ Módulo 0 listo.")
print("📌 Funciones disponibles: bgr_a_rgb, rgb_a_mp_image, cargar_imagen,")
print("   mostrar_imagen, mostrar_comparacion, calcular_angulo, landmark_a_pixeles")
print("\n💡 Nueva función clave: rgb_a_mp_image()")
print("   Convierte NumPy RGB → mp.Image (requerido por la nueva API)")


⬇️  Descargando modelos MediaPipe Tasks...
   ⬇️  Descargando face_landmarker.task... ✅ 3670 KB
   ⬇️  Descargando hand_landmarker.task... ✅ 7635 KB
   ⬇️  Descargando pose_landmarker_lite.task... ✅ 5642 KB
   ⬇️  Descargando pose_landmarker_full.task... ✅ 9177 KB
   ⬇️  Descargando pose_landmarker_heavy.task... ✅ 29945 KB
   ⬇️  Descargando gesture_recognizer.task... ✅ 8177 KB

✅ Módulo 0 listo.
📌 Funciones disponibles: bgr_a_rgb, rgb_a_mp_image, cargar_imagen,
   mostrar_imagen, mostrar_comparacion, calcular_angulo, landmark_a_pixeles

💡 Nueva función clave: rgb_a_mp_image()
   Convierte NumPy RGB → mp.Image (requerido por la nueva API)


---
# 😊 MÓDULO 1 — Face Mesh: 468 Landmarks del Rostro

## ¿Qué es Face Mesh?

MediaPipe Face Mesh detecta **468 puntos tridimensionales** en el rostro humano.  
Estos puntos cubren cada zona anatómica:

```
ZONAS PRINCIPALES:
  👁️  Ojo izquierdo   → landmarks 33, 133, 159, 145, 153, 144
  👁️  Ojo derecho     → landmarks 362, 263, 386, 374, 380, 373
  👃  Nariz           → landmarks 1, 2, 98, 327
  👄  Labios          → landmarks 13, 14, 17, 18, 61, 291, 78, 308
  🧔  Contorno cara   → landmarks 10, 338, 297, 332, 284, 251, 389...
  🤨  Cejas           → landmarks 70, 63, 105, 66, 107, 336...
```

## Aplicaciones
- 😴 Detección de somnolencia (ojos cerrados)  
- 🗣️ Detección de boca abierta / habla  
- 🚗 Sistemas de alerta para conductores  
- 🎭 Filtros de realidad aumentada  
- 😶 Análisis de expresiones faciales  

In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.A — MAPA DE LANDMARKS DEL ROSTRO                     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Definimos diccionarios con los índices de los landmarks más importantes.
# Esto nos permite referenciarlos por NOMBRE en lugar de número,
# haciendo el código mucho más legible y mantenible.

# ── Puntos clave del rostro (índices según MediaPipe Face Mesh) ───
#
# REFERENCIA VISUAL: https://mediapipe.dev/solutions/face_mesh
# Los índices son FIJOS: siempre el mismo número = el mismo punto anatómico.

FACE_LANDMARKS = {
    # ── OJOS ──────────────────────────────────────────────────────
    # Puntos que forman la apertura del ojo (borde del párpado)
    # Útiles para calcular EAR (Eye Aspect Ratio) → detectar somnolencia
    'ojo_izq_arriba':    159,   # párpado superior ojo izquierdo
    'ojo_izq_abajo':     145,   # párpado inferior ojo izquierdo
    'ojo_izq_izquierda':  33,   # comisura izquierda del ojo izquierdo
    'ojo_izq_derecha':   133,   # comisura derecha del ojo izquierdo
    'ojo_der_arriba':    386,   # párpado superior ojo derecho
    'ojo_der_abajo':     374,   # párpado inferior ojo derecho
    'ojo_der_izquierda': 362,   # comisura izquierda del ojo derecho
    'ojo_der_derecha':   263,   # comisura derecha del ojo derecho

    # ── BOCA ──────────────────────────────────────────────────────
    # Puntos del contorno de los labios
    # Útiles para MAR (Mouth Aspect Ratio) → detectar boca abierta / bostezo
    'labio_arriba':       13,   # labio superior (centro)
    'labio_abajo':        14,   # labio inferior (centro)
    'boca_izquierda':     61,   # comisura izquierda de la boca
    'boca_derecha':      291,   # comisura derecha de la boca

    # ── NARIZ ─────────────────────────────────────────────────────
    'punta_nariz':         1,   # punta de la nariz
    'nariz_puente':        6,   # puente de la nariz (entre los ojos)

    # ── CARA ──────────────────────────────────────────────────────
    'frente_centro':     151,   # centro de la frente
    'menton':            152,   # punta del mentón
    'mejilla_izquierda': 234,   # mejilla izquierda
    'mejilla_derecha':   454,   # mejilla derecha

    # ── CEJAS ─────────────────────────────────────────────────────
    'ceja_izq_centro':    70,   # punto central de la ceja izquierda
    'ceja_der_centro':   300,   # punto central de la ceja derecha
}

print("📍 Mapa de landmarks del rostro cargado.")
print(f"   Total de puntos clave catalogados: {len(FACE_LANDMARKS)}")
print()

# Mostramos la tabla organizada por zona
df_face = pd.DataFrame([
    {'Zona': 'Ojo',   'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if 'ojo' in k
] + [
    {'Zona': 'Boca',  'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if 'labio' in k or 'boca' in k
] + [
    {'Zona': 'Nariz', 'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if 'nariz' in k
] + [
    {'Zona': 'Cara',  'Nombre': k, 'Índice MediaPipe': v}
    for k, v in FACE_LANDMARKS.items() if k not in
    [k2 for k2 in FACE_LANDMARKS if 'ojo' in k2 or 'labio' in k2 or 'boca' in k2 or 'nariz' in k2]
])

print("📊 Tabla de landmarks clave del rostro:")
display(df_face)

📍 Mapa de landmarks del rostro cargado.
   Total de puntos clave catalogados: 20

📊 Tabla de landmarks clave del rostro:


,Zona,Nombre,Índice MediaPipe
0,Ojo,ojo_izq_arriba,159
1,Ojo,ojo_izq_abajo,145
2,Ojo,ojo_izq_izquierda,33
3,Ojo,ojo_izq_derecha,133
4,Ojo,ojo_der_arriba,386
5,Ojo,ojo_der_abajo,374
6,Ojo,ojo_der_izquierda,362
7,Ojo,ojo_der_derecha,263
8,Boca,labio_arriba,13
9,Boca,labio_abajo,14


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.B — DETECCIÓN DE FACE MESH EN IMAGEN ESTÁTICA        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# CAMBIOS API LEGACY → NUEVA API (mp.tasks):
#
#   Legacy:                              Nueva API:
#   ─────────────────────────────────    ─────────────────────────────────────
#   mp.solutions.face_mesh.FaceMesh(     vision.FaceLandmarkerOptions(
#     static_image_mode=True,              base_options=BaseOptions(model_asset_path=...),
#     max_num_faces=1,                     running_mode=RunningMode.IMAGE,
#     refine_landmarks=True,               num_faces=1,
#     min_detection_confidence=0.5         output_face_blendshapes=False,
#   )                                    )
#   detector.process(imagen_rgb)         detector.detect(mp.Image(...))
#   result.multi_face_landmarks[i]       result.face_landmarks[i]     ← lista plana
#   result.multi_face_landmarks[i]       result.face_landmarks[i][j]  ← landmark j
#     .landmark[j].x / .y / .z            .x / .y / .z                ← igual ✅


def analizar_rostro_imagen(imagen_rgb, dibujar_malla=True, dibujar_puntos_clave=True):
    """
    Detecta y visualiza los 468 landmarks del rostro en una imagen.
    
    Parámetros:
        imagen_rgb           → array NumPy RGB
        dibujar_malla        → si True, dibuja la red completa de 468 puntos
        dibujar_puntos_clave → si True, resalta puntos anatómicos importantes
    
    Retorna:
        imagen_anotada → imagen RGB con los landmarks dibujados
        resultado      → objeto FaceLandmarkerResult con todos los datos
        tabla_datos    → DataFrame con coordenadas de puntos clave
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho = imagen_rgb.shape[:2]

    # ── Configurar el detector ────────────────────────────────────
    opciones = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_FACE),
        # running_mode=RunningMode.IMAGE es el valor por defecto.
        # Otros modos: VIDEO (para video con timestamps) y LIVE_STREAM
        # (para cámara en tiempo real con callback asíncrono).
        running_mode=RunningMode.IMAGE,
        num_faces=1,                      # máximo de rostros a detectar
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=False,    # 52 blendshapes para expresiones (ARKit)
    )

    with FaceLandmarker.create_from_options(opciones) as detector:

        # ── Convertir imagen al formato de la nueva API ────────────
        # La nueva API NO acepta NumPy directamente.
        # Necesita un objeto mp.Image creado con rgb_a_mp_image().
        mp_imagen = rgb_a_mp_image(imagen_rgb)

        # ── Ejecutar detección ─────────────────────────────────────
        # detect() reemplaza a process() de la API legacy.
        resultado = detector.detect(mp_imagen)

        # ── Verificar si se detectó algún rostro ──────────────────
        # En la nueva API: result.face_landmarks es una lista de listas.
        # result.face_landmarks[i]    → landmarks del rostro i
        # result.face_landmarks[i][j] → landmark j del rostro i
        #
        # En la API legacy: result.multi_face_landmarks[i].landmark[j]
        if not resultado.face_landmarks:
            print("⚠️  No se detectó ningún rostro en la imagen.")
            return imagen_anotada, None, pd.DataFrame()

        print(f"✅ {len(resultado.face_landmarks)} rostro(s) detectado(s).")

        # ── Procesar cada rostro ───────────────────────────────────
        for idx_rostro, face_lms in enumerate(resultado.face_landmarks):
            # face_lms es una lista plana de 468 NormalizedLandmark
            # face_lms[j].x, face_lms[j].y, face_lms[j].z

            if dibujar_malla:
                # draw_landmarks() de la nueva API tiene la misma firma visual,
                # pero acepta una lista de NormalizedLandmark (no un LandmarkList).
                mp_drawing.draw_landmarks(
                    image          = imagen_anotada,
                    landmark_list  = face_lms,
                    connections    = FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                    # FACE_LANDMARKS_TESSELATION → malla densa completa (468 pts)
                    # FACE_LANDMARKS_CONTOURS    → solo el contorno
                    # FACE_LANDMARKS_FACE_OVAL   → óvalo del rostro
                    landmark_drawing_spec=mp_drawing.DrawingSpec(
                        color=(0, 255, 0), thickness=1, circle_radius=1
                    ),
                    connection_drawing_spec=mp_drawing.DrawingSpec(
                        color=(0, 200, 100), thickness=1
                    ),
                )
                # Contorno con estilo predefinido
                mp_drawing.draw_landmarks(
                    image          = imagen_anotada,
                    landmark_list  = face_lms,
                    connections    = FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style(),
                )

            if dibujar_puntos_clave:
                for nombre, idx in FACE_LANDMARKS.items():
                    lm = face_lms[idx]   # acceso directo por índice (lista plana)
                    px, py = landmark_a_pixeles(lm, alto, ancho)
                    cv2.circle(imagen_anotada, (px, py), 4, (255, 50, 50), -1)
                    cv2.putText(imagen_anotada, nombre[:8], (px+5, py),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.25, (255, 255, 0), 1)

            # Tabla de coordenadas
            filas = []
            for nombre, idx in FACE_LANDMARKS.items():
                lm = face_lms[idx]
                px, py = landmark_a_pixeles(lm, alto, ancho)
                filas.append({
                    'Nombre':       nombre,
                    'Índice MP':    idx,
                    'x (norm)':     round(lm.x, 4),
                    'y (norm)':     round(lm.y, 4),
                    'z (profund)':  round(lm.z, 4),
                    'x (píxeles)':  px,
                    'y (píxeles)':  py,
                })
            tabla_datos = pd.DataFrame(filas)

    return imagen_anotada, resultado, tabla_datos


print("✅ analizar_rostro_imagen() definida (API mp.tasks).")
print()
print("📌 Cambio clave respecto a la API legacy:")
print("   Legacy: result.multi_face_landmarks[i].landmark[j].x")
print("   Nueva:  result.face_landmarks[i][j].x")


✅ analizar_rostro_imagen() definida (API mp.tasks).

📌 Cambio clave respecto a la API legacy:
   Legacy: result.multi_face_landmarks[i].landmark[j].x
   Nueva:  result.face_landmarks[i][j].x


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.C — SUBIR IMAGEN Y ANALIZAR ROSTRO                   ║
# ╚══════════════════════════════════════════════════════════════════╝

uploader_rostro = widgets.FileUpload(
    accept='.jpg,.jpeg,.png', multiple=False, description='📷 Subir foto de rostro'
)
btn_analizar_rostro = widgets.Button(description='Analizar Rostro', button_style='success')
salida_rostro = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen con un rostro frontal claro:</b>'),
    uploader_rostro,
    btn_analizar_rostro,
    salida_rostro
]))


def on_analizar_rostro(_):
    with salida_rostro:
        clear_output()
        if not uploader_rostro.value:
            print("⚠️  Primero sube una imagen.")
            return

        # Extraer bytes del archivo subido (compatible con varias versiones de ipywidgets)
        valor = uploader_rostro.value
        archivo = valor[0] if isinstance(valor, (list, tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content

        # Convertir bytes → PIL → NumPy RGB
        imagen_pil = Image.open(io.BytesIO(contenido)).convert('RGB')
        imagen_rgb = np.array(imagen_pil)

        print(f"📐 Dimensiones: {imagen_rgb.shape[1]}×{imagen_rgb.shape[0]} px")
        print()

        # Ejecutar análisis
        imagen_anotada, resultados, tabla = analizar_rostro_imagen(
            imagen_rgb,
            dibujar_malla=True,
            dibujar_puntos_clave=True
        )

        # Visualizar comparación
        mostrar_comparacion(
            imagen_rgb, imagen_anotada,
            'Original', 'Face Mesh — 468 landmarks'
        )

        if not tabla.empty:
            print("\n📊 Coordenadas de puntos anatómicos clave:")
            display(tabla)


btn_analizar_rostro.on_click(on_analizar_rostro)

In [8]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 1.D — EAR y MAR: Detectar Somnolencia y Boca Abierta   ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# EAR (Eye Aspect Ratio) y MAR (Mouth Aspect Ratio) no cambian
# matemáticamente. Solo cambia cómo accedemos a los landmarks:
#
#   Legacy: landmarks[i].x  donde landmarks = face_landmarks.landmark
#   Nueva:  face_lms[i].x   donde face_lms  = result.face_landmarks[0]
#                                             (lista plana de NormalizedLandmark)

def calcular_ear(face_lms, puntos_ojo):
    """
    Eye Aspect Ratio para un ojo. Detecta somnolencia (EAR < 0.20).
    
    Parámetros:
        face_lms   → lista plana de landmarks (result.face_landmarks[0])
        puntos_ojo → 6 índices [izq, arriba1, arriba2, der, abajo1, abajo2]
    """
    # Acceso directo por índice: face_lms[i].x / .y
    p = [np.array([face_lms[i].x, face_lms[i].y]) for i in puntos_ojo]

    dist_v1 = np.linalg.norm(p[1] - p[5])
    dist_v2 = np.linalg.norm(p[2] - p[4])
    dist_h  = np.linalg.norm(p[0] - p[3])

    if dist_h == 0: return 0.0
    return round((dist_v1 + dist_v2) / (2.0 * dist_h), 3)


def calcular_mar(face_lms):
    """
    Mouth Aspect Ratio. Boca abierta cuando MAR > 0.50.
    
    Parámetros:
        face_lms → lista plana de landmarks (result.face_landmarks[0])
    """
    labio_sup = np.array([face_lms[FACE_LANDMARKS['labio_arriba']].x,
                          face_lms[FACE_LANDMARKS['labio_arriba']].y])
    labio_inf = np.array([face_lms[FACE_LANDMARKS['labio_abajo']].x,
                          face_lms[FACE_LANDMARKS['labio_abajo']].y])
    boca_izq  = np.array([face_lms[FACE_LANDMARKS['boca_izquierda']].x,
                          face_lms[FACE_LANDMARKS['boca_izquierda']].y])
    boca_der  = np.array([face_lms[FACE_LANDMARKS['boca_derecha']].x,
                          face_lms[FACE_LANDMARKS['boca_derecha']].y])

    dist_v = np.linalg.norm(labio_sup - labio_inf)
    dist_h = np.linalg.norm(boca_izq  - boca_der)

    if dist_h == 0: return 0.0
    return round(dist_v / dist_h, 3)


def analizar_expresiones_faciales(imagen_rgb):
    """
    Detecta rostro y calcula EAR + MAR.
    
    Cambio respecto a la API legacy:
      face_mesh.process(img)    →  detector.detect(mp.Image(...))
      result.multi_face_landmarks[0].landmark   →  result.face_landmarks[0]
    """
    PUNTOS_OJO_IZQ = [33,  160, 158, 133, 153, 144]
    PUNTOS_OJO_DER = [362, 385, 387, 263, 373, 380]

    opciones = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_FACE),
        running_mode=RunningMode.IMAGE,
        num_faces=1, min_face_detection_confidence=0.5,
    )

    with FaceLandmarker.create_from_options(opciones) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        if not resultado.face_landmarks:
            print("⚠️  No se detectó rostro.")
            return

        # Nueva API: resultado.face_landmarks[0] es la lista plana de 468 landmarks
        face_lms = resultado.face_landmarks[0]

        ear_izq = calcular_ear(face_lms, PUNTOS_OJO_IZQ)
        ear_der = calcular_ear(face_lms, PUNTOS_OJO_DER)
        ear_avg = round((ear_izq + ear_der) / 2, 3)
        mar     = calcular_mar(face_lms)

    print("=" * 50)
    print("  📊 ANÁLISIS DE EXPRESIÓN FACIAL")
    print("=" * 50)
    print(f"  EAR ojo izquierdo : {ear_izq}")
    print(f"  EAR ojo derecho   : {ear_der}")
    print(f"  EAR promedio      : {ear_avg}")
    print(f"  MAR boca          : {mar}")
    print("─" * 50)

    estado_ojos = "😴 CERRADOS (posible somnolencia)" if ear_avg < 0.20 else "👀 ABIERTOS"
    estado_boca = "😮 ABIERTA (bostezo/habla)"        if mar     > 0.50  else "😐 CERRADA"
    print(f"  Estado ojos : {estado_ojos}")
    print(f"  Estado boca : {estado_boca}")
    print("=" * 50)

    fig, ax = plt.subplots(figsize=(8, 4))
    metricas = ['EAR ojo izq', 'EAR ojo der', 'EAR promedio', 'MAR boca']
    valores  = [ear_izq, ear_der, ear_avg, mar]
    colores  = ['green' if ear_izq >= 0.20 else 'red',
                'green' if ear_der >= 0.20 else 'red',
                'green' if ear_avg >= 0.20 else 'red',
                'green' if mar     <= 0.50 else 'orange']
    barras = ax.bar(metricas, valores, color=colores, edgecolor='black', alpha=0.8)
    ax.axhline(y=0.20, color='red',    linestyle='--', alpha=0.7, label='Umbral EAR (0.20)')
    ax.axhline(y=0.50, color='orange', linestyle='--', alpha=0.7, label='Umbral MAR (0.50)')
    for barra, val in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.01,
               f'{val}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 0.8)
    ax.set_title('EAR y MAR — Análisis de apertura ocular y bucal', fontsize=13)
    ax.set_ylabel('Ratio'); ax.legend(); plt.tight_layout(); plt.show()


print("✅ calcular_ear(), calcular_mar(), analizar_expresiones_faciales() definidas.")
print()
print("📌 Cambio en el acceso a landmarks:")
print("   Legacy: result.multi_face_landmarks[0].landmark[i].x")
print("   Nueva:  result.face_landmarks[0][i].x")


✅ calcular_ear(), calcular_mar(), analizar_expresiones_faciales() definidas.

📌 Cambio en el acceso a landmarks:
   Legacy: result.multi_face_landmarks[0].landmark[i].x
   Nueva:  result.face_landmarks[0][i].x


---
# ✋ MÓDULO 2 — Hand Landmarks: 21 Puntos por Mano

## Anatomía de los landmarks de la mano

```
MediaPipe detecta 21 puntos por mano:

                     8   12  16  20
                     |    |   |   |
                  7  |  11|  15| 19|
                  |  6  10  14  18
            4     |  |   |   |   |
            |     5  9  13  17
            3      \ |   |  /
            |       \|   | /
            2        MUÑECA
            |           |
            1           0  ← WRIST (muñeca)

DEDO:    Pulgar   Índice  Medio  Anular  Meñique
BASE:      1        5      9      13      17
MED_1:     2        6      10     14      18
MED_2:     3        7      11     15      19
PUNTA:     4        8      12     16      20
```

## Índices a memorizar

| Punto | Nombre | Landmark # |
|---|---|---|
| Muñeca | WRIST | 0 |
| Punta pulgar | THUMB_TIP | 4 |
| Punta índice | INDEX_FINGER_TIP | 8 |
| Punta medio | MIDDLE_FINGER_TIP | 12 |
| Punta anular | RING_FINGER_TIP | 16 |
| Punta meñique | PINKY_TIP | 20 |

In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 2.A — MAPA DE LANDMARKS DE LA MANO                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# MediaPipe define constantes con los nombres de cada landmark.
# Podemos acceder a ellas como mp_hands.HandLandmark.WRIST, etc.
# Para mayor claridad, creamos un diccionario con nombres en español.

HAND_LANDMARKS = {
    # ── MUÑECA ────────────────────────────────────────────────────
    'muneca':              0,   # WRIST — base de la mano

    # ── PULGAR (THUMB) ────────────────────────────────────────────
    'pulgar_cmc':          1,   # carpometacarpal (unión con la palma)
    'pulgar_mcp':          2,   # metacarpofalángica
    'pulgar_ip':           3,   # interfalángica
    'pulgar_punta':        4,   # THUMB_TIP — punta del pulgar

    # ── ÍNDICE (INDEX) ────────────────────────────────────────────
    'indice_base':         5,   # metacarpofalángica del índice
    'indice_med1':         6,
    'indice_med2':         7,
    'indice_punta':        8,   # INDEX_FINGER_TIP

    # ── MEDIO (MIDDLE) ────────────────────────────────────────────
    'medio_base':          9,
    'medio_med1':         10,
    'medio_med2':         11,
    'medio_punta':        12,   # MIDDLE_FINGER_TIP

    # ── ANULAR (RING) ─────────────────────────────────────────────
    'anular_base':        13,
    'anular_med1':        14,
    'anular_med2':        15,
    'anular_punta':       16,   # RING_FINGER_TIP

    # ── MEÑIQUE (PINKY) ───────────────────────────────────────────
    'menique_base':       17,
    'menique_med1':       18,
    'menique_med2':       19,
    'menique_punta':      20,   # PINKY_TIP
}

# Agrupación de puntas de dedos (muy útil para detección de gestos)
PUNTAS_DEDOS = {
    'pulgar':  4,
    'indice':  8,
    'medio':  12,
    'anular': 16,
    'menique':20,
}

# Bases de los dedos (articulaciones MCP)
BASES_DEDOS = {
    'pulgar':  2,   # MCP del pulgar
    'indice':  5,
    'medio':   9,
    'anular': 13,
    'menique':17,
}

print("✅ Mapa de landmarks de la mano cargado.")
print(f"   Total de landmarks: {len(HAND_LANDMARKS)}")
print()

# Visualización del esquema de la mano
print("📌 ESQUEMA DE CONEXIONES DE MEDIAPIPE HANDS:")
print()
conexiones_texto = [
    ("Palma",     "0-1-2-3-4     (muñeca → pulgar)"),
    ("Índice",    "0-5-6-7-8     (muñeca → punta índice)"),
    ("Medio",     "0-9-10-11-12  (muñeca → punta medio)"),
    ("Anular",    "0-13-14-15-16 (muñeca → punta anular)"),
    ("Meñique",   "0-17-18-19-20 (muñeca → punta meñique)"),
    ("Transversal","5-9-13-17    (bases de los dedos)"),
]
for nombre, conexion in conexiones_texto:
    print(f"   {nombre:<12} → {conexion}")

✅ Mapa de landmarks de la mano cargado.
   Total de landmarks: 21

📌 ESQUEMA DE CONEXIONES DE MEDIAPIPE HANDS:

   Palma        → 0-1-2-3-4     (muñeca → pulgar)
   Índice       → 0-5-6-7-8     (muñeca → punta índice)
   Medio        → 0-9-10-11-12  (muñeca → punta medio)
   Anular       → 0-13-14-15-16 (muñeca → punta anular)
   Meñique      → 0-17-18-19-20 (muñeca → punta meñique)
   Transversal  → 5-9-13-17    (bases de los dedos)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 2.B — DETECCIÓN DE MANOS EN IMAGEN                     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# CAMBIOS API LEGACY → NUEVA API:
#
#   Legacy:                                 Nueva API:
#   ──────────────────────────────────      ───────────────────────────────────────
#   mp_hands.Hands(                         HandLandmarkerOptions(
#     max_num_hands=2,                        base_options=BaseOptions(...),
#     min_detection_confidence=0.5            num_hands=2,
#   )                                         min_hand_detection_confidence=0.5,
#                                           )
#
#   result.multi_hand_landmarks[i]          result.hand_landmarks[i]
#     (LandmarkList con .landmark[j])         (lista plana de NormalizedLandmark)
#
#   result.multi_handedness[i]              result.handedness[i]
#     .classification[0].label               [0].category_name   ← cambia el campo
#     .classification[0].score               [0].score           ← igual ✅
#
#   mp_hands.HAND_CONNECTIONS               HandLandmarksConnections.HAND_CONNECTIONS
#   mp_drawing_styles                       mp_drawing_styles
#     .get_default_hand_landmarks_style()     .get_default_hand_landmarks_style()  ✅

def analizar_manos_imagen(imagen_rgb, max_manos=2):
    """
    Detecta y visualiza los 21 landmarks de cada mano.
    
    Retorna:
        imagen_anotada → imagen con landmarks dibujados
        resultado      → HandLandmarkerResult completo
        info_manos     → lista de dicts con datos por mano
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]
    info_manos     = []

    opciones = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_HAND),
        running_mode=RunningMode.IMAGE,
        num_hands=max_manos,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    with HandLandmarker.create_from_options(opciones) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        if not resultado.hand_landmarks:
            print("⚠️  No se detectaron manos en la imagen.")
            return imagen_anotada, None, []

        print(f"✅ {len(resultado.hand_landmarks)} mano(s) detectada(s).")

        # ── Iterar sobre cada mano detectada ──────────────────────
        # Nueva API: resultado.hand_landmarks[i] y resultado.handedness[i]
        # son listas paralelas (mismo índice = misma mano).
        for idx_mano in range(len(resultado.hand_landmarks)):
            hand_lms   = resultado.hand_landmarks[idx_mano]   # lista plana de 21 lms
            handedness = resultado.handedness[idx_mano]       # lista de Category

            # En la API legacy: handedness.classification[0].label
            # En la nueva API:  handedness[0].category_name
            etiqueta  = handedness[0].category_name   # 'Left' o 'Right'
            confianza = handedness[0].score

            print(f"   Mano {idx_mano+1}: {etiqueta} ({confianza:.1%} confianza)")

            # Dibujar esqueleto de la mano
            mp_drawing.draw_landmarks(
                image         = imagen_anotada,
                landmark_list = hand_lms,
                connections   = HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_hand_landmarks_style(),
                connection_drawing_spec=mp_drawing_styles.get_default_hand_connections_style(),
            )

            # Resaltar puntas de dedos
            for nombre_dedo, idx_lm in PUNTAS_DEDOS.items():
                lm = hand_lms[idx_lm]   # acceso directo por índice
                px, py = landmark_a_pixeles(lm, alto, ancho)
                cv2.circle(imagen_anotada, (px, py), 10, (255, 255, 255), 2)
                cv2.circle(imagen_anotada, (px, py),  6, (0, 200, 255), -1)
                cv2.putText(imagen_anotada, nombre_dedo[:4], (px-10, py-12),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)

            # Numerar los 21 landmarks
            for j, lm in enumerate(hand_lms):
                px, py = landmark_a_pixeles(lm, alto, ancho)
                cv2.putText(imagen_anotada, str(j), (px+4, py-4),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.3, (200, 200, 200), 1)

            # Tabla de coordenadas
            filas = [{'Mano': etiqueta, 'Punto': nombre, 'Índice': idx_lm,
                      'x (norm)': round(hand_lms[idx_lm].x, 4),
                      'y (norm)': round(hand_lms[idx_lm].y, 4),
                      'z (prof)': round(hand_lms[idx_lm].z, 4),
                      'x (px)': landmark_a_pixeles(hand_lms[idx_lm], alto, ancho)[0],
                      'y (px)': landmark_a_pixeles(hand_lms[idx_lm], alto, ancho)[1]}
                     for nombre, idx_lm in HAND_LANDMARKS.items()]

            info_manos.append({
                'etiqueta':  etiqueta,
                'confianza': confianza,
                'landmarks': hand_lms,
                'tabla':     pd.DataFrame(filas),
            })

    return imagen_anotada, resultado, info_manos


# Widget
uploader_mano = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='✋ Subir foto de mano')
btn_analizar_mano = widgets.Button(description='Analizar Mano', button_style='info')
salida_mano = widgets.Output()
display(widgets.VBox([widgets.HTML('<b>Sube una imagen con una o dos manos:</b>'),
                      uploader_mano, btn_analizar_mano, salida_mano]))

def on_analizar_mano(_):
    with salida_mano:
        clear_output()
        if not uploader_mano.value: print("⚠️  Primero sube una imagen."); return
        valor = uploader_mano.value
        archivo = valor[0] if isinstance(valor, (list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo, dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada, resultado, info = analizar_manos_imagen(img_rgb, max_manos=2)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Hand Landmarks — 21 pts/mano')
        for mano in info:
            print(f"\n📊 Coordenadas — Mano {mano['etiqueta']}:"); display(mano['tabla'].head(10))

btn_analizar_mano.on_click(on_analizar_mano)
print("✅ analizar_manos_imagen() definida.")
print()
print("📌 Cambios clave:")
print("   handedness.classification[0].label  →  handedness[0].category_name")
print("   result.multi_hand_landmarks[i]       →  result.hand_landmarks[i]")
print("   mp_hands.HAND_CONNECTIONS            →  HandLandmarksConnections.HAND_CONNECTIONS")


✅ analizar_manos_imagen() definida.

📌 Cambios clave:
   handedness.classification[0].label  →  handedness[0].category_name
   result.multi_hand_landmarks[i]       →  result.hand_landmarks[i]
   mp_hands.HAND_CONNECTIONS            →  HandLandmarksConnections.HAND_CONNECTIONS


---
# 🧍 MÓDULO 3 — Body Pose: 33 Landmarks del Cuerpo

## Landmarks del cuerpo humano

```
MediaPipe Pose detecta 33 puntos del cuerpo:

         0(nariz)
    1  2  3  4    ← cara (ojos, orejas)
       5  6       ← hombros
      7    8      ← codos
     9     10     ← muñecas
    11     12     ← dedos (pulgares y meñiques)
       13 14      ← caderas
       15 16      ← rodillas
       17 18      ← tobillos
      19   20     ← talones
      21   22     ← puntas de pies

IZQUIERDA = números impares
DERECHA   = números pares
```

## Articulaciones clave para análisis biomecánico

| Articulación | Landmarks | Ángulo posible |
|---|---|---|
| Codo izquierdo | 11-13-15 | Flexión del brazo |
| Codo derecho | 12-14-16 | Flexión del brazo |
| Rodilla izquierda | 23-25-27 | Flexión de pierna |
| Rodilla derecha | 24-26-28 | Flexión de pierna |
| Cadera izquierda | 11-23-25 | Flexión de cadera |
| Hombro izquierdo | 13-11-23 | Abducción del brazo |

In [11]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 3.A — MAPA DE LANDMARKS DEL CUERPO                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# Los 33 landmarks de MediaPipe Pose con nombres descriptivos
POSE_LANDMARKS = {
    # ── CABEZA / CARA ──────────────────────────────────────────────
    'nariz':                0,
    'ojo_izq_inner':        1,   # esquina interna del ojo izquierdo
    'ojo_izq':              2,   # centro del ojo izquierdo
    'ojo_izq_outer':        3,   # esquina externa del ojo izquierdo
    'ojo_der_inner':        4,
    'ojo_der':              5,
    'ojo_der_outer':        6,
    'oreja_izq':            7,
    'oreja_der':            8,
    'boca_izq':             9,
    'boca_der':            10,

    # ── TREN SUPERIOR ─────────────────────────────────────────────
    'hombro_izq':          11,   # articulación del hombro izquierdo
    'hombro_der':          12,
    'codo_izq':            13,
    'codo_der':            14,
    'muneca_izq':          15,
    'muneca_der':          16,
    'pulgar_izq':          17,   # articulación del pulgar
    'pulgar_der':          18,
    'indice_izq':          19,   # punta del dedo índice
    'indice_der':          20,
    'menique_izq':         21,   # punta del meñique
    'menique_der':         22,

    # ── TREN INFERIOR ─────────────────────────────────────────────
    'cadera_izq':          23,   # articulación de la cadera izquierda
    'cadera_der':          24,
    'rodilla_izq':         25,
    'rodilla_der':         26,
    'tobillo_izq':         27,
    'tobillo_der':         28,
    'talon_izq':           29,
    'talon_der':           30,
    'pie_izq':             31,   # punta del pie izquierdo
    'pie_der':             32,
}

# Articulaciones definidas para cálculo de ángulos biomecánicos
# Cada entrada: (punto_A, punto_B_vértice, punto_C)
ANGULOS_BIOMECÁNICOS = {
    'codo_izquierdo':    (11, 13, 15),   # hombro-codo-muñeca
    'codo_derecho':      (12, 14, 16),
    'hombro_izquierdo':  (13, 11, 23),   # codo-hombro-cadera
    'hombro_derecho':    (14, 12, 24),
    'cadera_izquierda':  (11, 23, 25),   # hombro-cadera-rodilla
    'cadera_derecha':    (12, 24, 26),
    'rodilla_izquierda': (23, 25, 27),   # cadera-rodilla-tobillo
    'rodilla_derecha':   (24, 26, 28),
    'tobillo_izquierdo': (25, 27, 31),   # rodilla-tobillo-pie
    'tobillo_derecho':   (26, 28, 32),
}

print("✅ Mapa de landmarks del cuerpo cargado.")
print(f"   Total landmarks: {len(POSE_LANDMARKS)}")
print(f"   Ángulos biomecánicos definidos: {len(ANGULOS_BIOMECÁNICOS)}")
print()

# Tabla organizada por zona corporal
zonas = {
    'Cabeza/Cara': [k for k in POSE_LANDMARKS if any(p in k for p in ['nariz','ojo','oreja','boca'])],
    'Tren Superior': [k for k in POSE_LANDMARKS if any(p in k for p in ['hombro','codo','muneca','pulgar','indice','menique'])],
    'Tren Inferior': [k for k in POSE_LANDMARKS if any(p in k for p in ['cadera','rodilla','tobillo','talon','pie'])],
}

for zona, puntos in zonas.items():
    print(f"  📍 {zona}:")
    for p in puntos:
        print(f"      [{POSE_LANDMARKS[p]:>2}] {p}")
    print()

✅ Mapa de landmarks del cuerpo cargado.
   Total landmarks: 33
   Ángulos biomecánicos definidos: 10

  📍 Cabeza/Cara:
      [ 0] nariz
      [ 1] ojo_izq_inner
      [ 2] ojo_izq
      [ 3] ojo_izq_outer
      [ 4] ojo_der_inner
      [ 5] ojo_der
      [ 6] ojo_der_outer
      [ 7] oreja_izq
      [ 8] oreja_der
      [ 9] boca_izq
      [10] boca_der

  📍 Tren Superior:
      [11] hombro_izq
      [12] hombro_der
      [13] codo_izq
      [14] codo_der
      [15] muneca_izq
      [16] muneca_der
      [17] pulgar_izq
      [18] pulgar_der
      [19] indice_izq
      [20] indice_der
      [21] menique_izq
      [22] menique_der

  📍 Tren Inferior:
      [23] cadera_izq
      [24] cadera_der
      [25] rodilla_izq
      [26] rodilla_der
      [27] tobillo_izq
      [28] tobillo_der
      [29] talon_izq
      [30] talon_der
      [31] pie_izq
      [32] pie_der



In [12]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 3.B — DETECCIÓN DE POSE EN IMAGEN Y ÁNGULOS            ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# CAMBIOS API LEGACY → NUEVA API:
#
#   Legacy:                                 Nueva API:
#   ──────────────────────────────────      ───────────────────────────────────────
#   mp_pose.Pose(                           PoseLandmarkerOptions(
#     static_image_mode=True,                 base_options=BaseOptions(model_asset_path=...),
#     model_complexity=1,  ← 0/1/2           running_mode=RunningMode.IMAGE,
#     min_detection_confidence=0.5           num_poses=1,
#   )                                         min_pose_detection_confidence=0.5,
#                                           )
#   ⚠️ model_complexity en la nueva API se elige por MODELO:
#       pose_landmarker_lite.task   → equivale a model_complexity=0
#       pose_landmarker_full.task   → equivale a model_complexity=1 ✅
#       pose_landmarker_heavy.task  → equivale a model_complexity=2
#
#   result.pose_landmarks.landmark[i]       result.pose_landmarks[0][i]
#     .x / .y / .z / .visibility              .x / .y / .z / .visibility  ✅
#
#   mp_pose.POSE_CONNECTIONS                PoseLandmarksConnections.POSE_LANDMARKS

def analizar_pose_imagen(imagen_rgb, calcular_angulos=True, modelo='full'):
    """
    Detecta 33 landmarks del cuerpo y calcula ángulos articulares.
    
    Parámetros:
        imagen_rgb       → array NumPy RGB
        calcular_angulos → True = calcula ángulos biomecánicos
        modelo           → 'lite' | 'full' | 'heavy'
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]
    angulos_dict   = {}

    # En la nueva API el modelo se elige por archivo, no por parámetro numérico.
    # Esto es más explícito: se sabe exactamente qué modelo se está usando.
    modelo_path = {
        'lite':  MODEL_POSE_LITE,
        'full':  MODEL_POSE_FULL,
        'heavy': MODEL_POSE_HEAVY,
    }.get(modelo, MODEL_POSE_FULL)

    opciones = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=modelo_path),
        running_mode=RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,  # genera máscara de la persona (más lento)
    )

    with PoseLandmarker.create_from_options(opciones) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        # Nueva API: resultado.pose_landmarks es una lista de listas.
        # resultado.pose_landmarks[0]    → lista plana de 33 NormalizedLandmark
        # resultado.pose_landmarks[0][i] → landmark i de la primera persona
        if not resultado.pose_landmarks:
            print("⚠️  No se detectó ninguna persona en la imagen.")
            return imagen_anotada, {}, pd.DataFrame()

        print("✅ Persona detectada.")

        # Landmarks de la primera persona (índice 0)
        landmarks = resultado.pose_landmarks[0]

        # Dibujar esqueleto con la nueva API
        mp_drawing.draw_landmarks(
            image         = imagen_anotada,
            landmark_list = landmarks,
            connections   = PoseLandmarksConnections.POSE_LANDMARKS,
            # ↑ Equivalente a mp_pose.POSE_CONNECTIONS de la API legacy
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(255, 255, 0), thickness=3
            ),
        )

        if calcular_angulos:
            for nombre, (idx_a, idx_b, idx_c) in ANGULOS_BIOMECÁNICOS.items():
                lm_a = landmarks[idx_a]
                lm_b = landmarks[idx_b]
                lm_c = landmarks[idx_c]

                # .visibility sigue disponible igual que en la API legacy
                if (lm_a.visibility > 0.5 and lm_b.visibility > 0.5
                        and lm_c.visibility > 0.5):
                    angulo = calcular_angulo([lm_a.x, lm_a.y],
                                             [lm_b.x, lm_b.y],
                                             [lm_c.x, lm_c.y])
                    angulos_dict[nombre] = angulo

                    px_b, py_b = landmark_a_pixeles(lm_b, alto, ancho)
                    texto = f"{angulo:.0f}°"
                    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(imagen_anotada,
                                 (px_b-5, py_b-th-10), (px_b+tw+5, py_b+5),
                                 (0,0,0), -1)
                    color_texto = (0,255,0) if 30 < angulo < 170 else (0,50,255)
                    cv2.putText(imagen_anotada, texto, (px_b, py_b-5),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2)

        # Tabla de coordenadas
        filas = [{'Punto': nombre, 'Índice': idx,
                  'x (norm)': round(landmarks[idx].x, 4),
                  'y (norm)': round(landmarks[idx].y, 4),
                  'z (prof)': round(landmarks[idx].z, 4),
                  'visibilidad': round(landmarks[idx].visibility, 3),
                  'x (px)': landmark_a_pixeles(landmarks[idx], alto, ancho)[0],
                  'y (px)': landmark_a_pixeles(landmarks[idx], alto, ancho)[1]}
                 for nombre, idx in POSE_LANDMARKS.items()]
        tabla_coords = pd.DataFrame(filas)

    return imagen_anotada, angulos_dict, tabla_coords


def visualizar_angulos(angulos_dict):
    """Gráfica de barras horizontales con los ángulos articulares."""
    if not angulos_dict: print("⚠️  No hay ángulos."); return
    nombres = list(angulos_dict.keys()); valores = list(angulos_dict.values())
    colores = ['#2196F3' if 'codo' in n else '#4CAF50' if 'rodilla' in n else
               '#FF9800' if 'hombro' in n else '#9C27B0' if 'cadera' in n else
               '#F44336' for n in nombres]
    fig, ax = plt.subplots(figsize=(12, 5))
    barras = ax.barh(nombres, valores, color=colores, edgecolor='white', alpha=0.85)
    for barra, val in zip(barras, valores):
        ax.text(val+1, barra.get_y()+barra.get_height()/2,
               f'{val:.1f}°', va='center', fontsize=10, fontweight='bold')
    ax.axvline(x=180, color='gray', linestyle='--', alpha=0.5, label='180° (extendido)')
    ax.axvline(x=90,  color='blue', linestyle='--', alpha=0.5, label='90° (recto)')
    ax.set_xlim(0, 200); ax.set_xlabel('Ángulo (grados)', fontsize=11)
    ax.set_title('📐 Ángulos Articulares Biomecánicos', fontsize=14, fontweight='bold')
    leyenda = [mpatches.Patch(color=c, label=l) for c, l in
               [('#2196F3','Codo'),('#4CAF50','Rodilla'),('#FF9800','Hombro'),('#9C27B0','Cadera')]]
    ax.legend(handles=leyenda, loc='lower right'); plt.tight_layout(); plt.show()


# Widget
uploader_pose = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🧍 Subir foto de cuerpo')
btn_analizar_pose = widgets.Button(description='Analizar Postura', button_style='warning')
salida_pose = widgets.Output()
display(widgets.VBox([widgets.HTML('<b>Sube una imagen de cuerpo completo o medio cuerpo:</b>'),
                      uploader_pose, btn_analizar_pose, salida_pose]))

def on_analizar_pose(_):
    with salida_pose:
        clear_output()
        if not uploader_pose.value: print("⚠️  Primero sube una imagen."); return
        valor = uploader_pose.value
        archivo = valor[0] if isinstance(valor,(list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo,dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada, angulos, tabla = analizar_pose_imagen(img_rgb, calcular_angulos=True)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Pose — 33 landmarks + ángulos')
        if angulos:
            print("\n📐 Ángulos articulares:")
            for a, g in angulos.items(): print(f"   {a:<25}: {g}°")
            visualizar_angulos(angulos)
        if not tabla.empty:
            print("\n📊 Coordenadas de landmarks:"); display(tabla)

btn_analizar_pose.on_click(on_analizar_pose)
print("✅ analizar_pose_imagen() definida.")
print()
print("📌 Cambios clave:")
print("   model_complexity=0/1/2  →  MODEL_POSE_LITE / FULL / HEAVY  (archivo .task)")
print("   result.pose_landmarks.landmark[i]  →  result.pose_landmarks[0][i]")
print("   mp_pose.POSE_CONNECTIONS           →  PoseLandmarksConnections.POSE_LANDMARKS")


✅ analizar_pose_imagen() definida.

📌 Cambios clave:
   model_complexity=0/1/2  →  MODEL_POSE_LITE / FULL / HEAVY  (archivo .task)
   result.pose_landmarks.landmark[i]  →  result.pose_landmarks[0][i]
   mp_pose.POSE_CONNECTIONS           →  PoseLandmarksConnections.POSE_LANDMARKS


---
# 🔮 MÓDULO 4 — Holistic: Rostro + Manos + Cuerpo Simultáneo

## ¿Qué es MediaPipe Holistic?

**Holistic** combina los tres modelos anteriores en una **única pasada de inferencia**,  
lo que lo hace más eficiente que ejecutarlos por separado.

```
Una sola imagen → UNA llamada a .process() → obtienes:
  ✅ 468 landmarks del rostro
  ✅ 33 landmarks del cuerpo
  ✅ 21 landmarks mano izquierda
  ✅ 21 landmarks mano derecha
  ───────────────────────────────
  TOTAL: 543 puntos simultáneos
```

In [13]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 4 — ANÁLISIS COMPLETO: ROSTRO + MANOS + CUERPO         ║
# ║             (Holistic reimplementado con la nueva API)          ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# ⚠️ CAMBIO IMPORTANTE: Holistic no existe en la nueva API
# ─────────────────────────────────────────────────────────────────
# La API legacy tenía mp.solutions.holistic.Holistic, que combinaba
# los tres modelos en UNA SOLA pasada de inferencia.
#
# La nueva API mp.tasks NO incluye Holistic.
# La solución equivalente es ejecutar los tres detectores EN SECUENCIA
# sobre la misma imagen. Hay una pequeña pérdida de eficiencia
# (3 llamadas en lugar de 1), pero el resultado visual es idéntico.
#
# Comparativa:
#   Legacy:  Holistic.process(img)   → 1 llamada, 543 landmarks
#   Nueva:   Face + Hand + Pose      → 3 llamadas, 543 landmarks
#
# En Colab (CPU), el tiempo extra es de ~50-100ms por imagen,
# lo cual es aceptable para análisis de imágenes estáticas.

def analizar_holistic_imagen(imagen_rgb):
    """
    Ejecuta detección simultánea de rostro, cuerpo y manos
    usando tres detectores de la nueva API mp.tasks.
    
    Retorna:
        imagen_anotada → imagen con todos los landmarks dibujados
        resumen        → dict con conteo de landmarks detectados
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]
    resumen        = {'rostro': 0, 'cuerpo': 0, 'mano_izq': 0, 'mano_der': 0}

    # Convertir la imagen UNA SOLA VEZ (se reutiliza en los 3 detectores)
    mp_imagen = rgb_a_mp_image(imagen_rgb)

    # ── 1. Detector de rostro ─────────────────────────────────────
    opts_face = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_FACE),
        running_mode=RunningMode.IMAGE, num_faces=1,
    )
    with FaceLandmarker.create_from_options(opts_face) as det_face:
        res_face = det_face.detect(mp_imagen)

    if res_face.face_landmarks:
        resumen['rostro'] = 468
        face_lms = res_face.face_landmarks[0]
        mp_drawing.draw_landmarks(
            imagen_anotada, face_lms,
            FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(80, 110, 10), thickness=1, circle_radius=1
            ),
        )
        mp_drawing.draw_landmarks(
            imagen_anotada, face_lms,
            FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(0, 255, 0), thickness=1
            ),
        )

    # ── 2. Detector de pose (cuerpo) ──────────────────────────────
    opts_pose = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_POSE_FULL),
        running_mode=RunningMode.IMAGE, num_poses=1,
    )
    with PoseLandmarker.create_from_options(opts_pose) as det_pose:
        res_pose = det_pose.detect(mp_imagen)

    if res_pose.pose_landmarks:
        resumen['cuerpo'] = 33
        mp_drawing.draw_landmarks(
            imagen_anotada, res_pose.pose_landmarks[0],
            PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(245, 117, 66), thickness=3
            ),
        )

    # ── 3. Detector de manos (izquierda y derecha) ─────────────────
    opts_hand = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_HAND),
        running_mode=RunningMode.IMAGE, num_hands=2,
    )
    with HandLandmarker.create_from_options(opts_hand) as det_hand:
        res_hand = det_hand.detect(mp_imagen)

    if res_hand.hand_landmarks:
        for i, (hand_lms, handedness) in enumerate(
            zip(res_hand.hand_landmarks, res_hand.handedness)
        ):
            # handedness[0].category_name → 'Left' o 'Right'
            lado = handedness[0].category_name
            color_mano = (0, 200, 255) if lado == 'Left' else (255, 100, 200)

            if lado == 'Left':
                resumen['mano_izq'] = 21
            else:
                resumen['mano_der'] = 21

            mp_drawing.draw_landmarks(
                imagen_anotada, hand_lms,
                HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_hand_landmarks_style(),
                connection_drawing_spec=mp_drawing.DrawingSpec(
                    color=color_mano, thickness=2
                ),
            )

    # Leyenda en la imagen
    leyenda_items = [
        (f"Rostro: {resumen['rostro']} pts",   (80, 255, 80)),
        (f"Cuerpo: {resumen['cuerpo']} pts",   (245, 117, 66)),
        (f"M.Izq:  {resumen['mano_izq']} pts", (0, 200, 255)),
        (f"M.Der:  {resumen['mano_der']} pts", (255, 100, 200)),
    ]
    for i, (texto, color) in enumerate(leyenda_items):
        cv2.rectangle(imagen_anotada, (10, 10+i*30), (200, 35+i*30), (0,0,0), -1)
        cv2.putText(imagen_anotada, texto, (15, 28+i*30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)

    return imagen_anotada, resumen


# Widget
uploader_holistic = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🔮 Subir imagen completa')
btn_holistic = widgets.Button(description='Análisis Completo', button_style='danger')
salida_holistic = widgets.Output()
display(widgets.VBox([widgets.HTML('<b>Sube una imagen que muestre rostro + cuerpo + manos:</b>'),
                      uploader_holistic, btn_holistic, salida_holistic]))

def on_holistic(_):
    with salida_holistic:
        clear_output()
        if not uploader_holistic.value: print("⚠️  Primero sube una imagen."); return
        valor = uploader_holistic.value
        archivo = valor[0] if isinstance(valor,(list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo,dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))

        print("🔄 Procesando con 3 detectores: rostro + cuerpo + manos...")
        img_anotada, resumen = analizar_holistic_imagen(img_rgb)

        total = sum(resumen.values())
        print(f"✅ Total de landmarks detectados: {total}")
        for zona, count in resumen.items():
            print(f"   {'✅' if count > 0 else '❌'} {zona:<12}: {count} puntos")
        mostrar_comparacion(img_rgb, img_anotada, 'Original',
                           f'Análisis completo — {total} landmarks')

btn_holistic.on_click(on_holistic)
print("✅ analizar_holistic_imagen() definida.")
print()
print("⚠️  Holistic reemplazado por 3 detectores separados (mp.tasks no incluye Holistic).")
print("   Face + Hand + Pose → mismo resultado visual, ~50ms adicionales por imagen.")


✅ analizar_holistic_imagen() definida.

⚠️  Holistic reemplazado por 3 detectores separados (mp.tasks no incluye Holistic).
   Face + Hand + Pose → mismo resultado visual, ~50ms adicionales por imagen.


---
# ✊ MÓDULO 5 — Detección de Gestos con la Mano

## ¿Cómo se detectan los gestos?

La estrategia es sencilla y poderosa:  
**compararemos la posición de las puntas de los dedos respecto a sus bases**.

```
Si la PUNTA del dedo está MÁS ARRIBA (y menor) que su BASE → dedo EXTENDIDO
Si la PUNTA del dedo está MÁS ABAJO  (y mayor) que su BASE → dedo DOBLADO

Recordatorio: en coordenadas de imagen, y=0 es ARRIBA, y=1 es ABAJO
```

## Gestos que detectaremos

| Gesto | Descripción |
|---|---|
| ✊ Puño | Todos los dedos doblados |
| ✋ Mano abierta | Todos los dedos extendidos |
| 👍 Pulgar arriba | Solo pulgar extendido |
| 👎 Pulgar abajo | Solo pulgar extendido hacia abajo |
| ☝️ Apuntando | Solo índice extendido |
| ✌️ Victoria / Paz | Índice y medio extendidos |
| 🤟 Te amo | Pulgar, índice y meñique |
| 🤙 Llámame | Pulgar y meñique extendidos |

In [14]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 5 — DETECCIÓN DE GESTOS DE LA MANO                     ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# MEJORA IMPORTANTE: GestureRecognizer en la nueva API
# ─────────────────────────────────────────────────────
# La API legacy no tenía reconocimiento nativo de gestos.
# La lógica de detectar_dedos_extendidos() + clasificar_gesto()
# era una solución manual basada en geometría de landmarks.
#
# La nueva API incluye GestureRecognizer, que:
#   ✅ Detecta gestos entrenados con ML (más robusto que reglas manuales)
#   ✅ Incluye gestos: None, Closed_Fist, Open_Palm, Pointing_Up,
#      Thumb_Down, Thumb_Up, Victory, ILoveYou
#   ✅ Devuelve la mano con sus landmarks Y el gesto reconocido
#
# Se mantiene la lógica manual (detectar_dedos_extendidos) como
# FALLBACK para gestos no cubiertos por el modelo entrenado.

# ── Funciones auxiliares (sin cambios, compatibles con nueva API) ─

def detectar_dedos_extendidos(hand_lms):
    """
    Determina cuáles dedos están extendidos.
    Ahora recibe la lista plana de landmarks de la nueva API.
    Antes: hand_landmarks.landmark (LandmarkList)
    Ahora: result.hand_landmarks[i]  (lista directa) — misma interfaz .x/.y
    """
    puntas = {d: hand_lms[idx] for d, idx in PUNTAS_DEDOS.items()}
    bases  = {d: hand_lms[idx] for d, idx in BASES_DEDOS.items()}

    dedos_ext = {}
    # Pulgar: eje X
    pulgar_punta = hand_lms[4]; pulgar_base = hand_lms[2]; muneca = hand_lms[0]
    dedos_ext['pulgar'] = abs(pulgar_punta.x - muneca.x) > abs(pulgar_base.x - muneca.x)
    # Otros dedos: eje Y
    for dedo in ['indice', 'medio', 'anular', 'menique']:
        dedos_ext[dedo] = puntas[dedo].y < bases[dedo].y

    return dedos_ext


def clasificar_gesto(dedos):
    """Clasifica gesto por patrón de dedos. Sin cambios respecto a la versión legacy."""
    p, i, m, a, me = dedos['pulgar'], dedos['indice'], dedos['medio'], dedos['anular'], dedos['menique']
    if not any([p,i,m,a,me]):   return '✊', 'Puño cerrado'
    if all([p,i,m,a,me]):       return '✋', 'Mano abierta'
    if p and not i and not m and not a and not me: return '👍', 'Pulgar arriba'
    if not p and i and not m and not a and not me: return '☝️', 'Apuntando'
    if not p and i and m and not a and not me:     return '✌️', 'Victoria'
    if not p and i and m and a and not me:         return '🤟', 'Tres dedos'
    if not p and i and m and a and me:             return '🖖', 'Cuatro dedos'
    if p and not i and not m and not a and me:     return '🤙', 'Llámame'
    if p and i and not m and not a and not me:     return '🤘', 'Rock'
    return '❓', f'{sum([p,i,m,a,me])} dedos'


def analizar_gestos_imagen(imagen_rgb):
    """
    Detecta manos con GestureRecognizer (nueva API).
    
    GestureRecognizer combina HandLandmarker + clasificador de gestos.
    Devuelve landmarks + gesto reconocido en una sola llamada.
    
    Cambios respecto a la versión legacy con mp_hands.Hands:
      result.multi_hand_landmarks  →  result.hand_landmarks
      result.multi_handedness      →  result.handedness
      (nuevo) result.gestures[i][0].category_name  → gesto ML reconocido
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]

    # GestureRecognizerOptions incluye todo:
    # detección de manos + landmarks + clasificación de gestos
    opts = GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_GESTURE),
        running_mode=RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.6,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    with GestureRecognizer.create_from_options(opts) as recognizer:
        resultado = recognizer.recognize(rgb_a_mp_image(imagen_rgb))
        # GestureRecognizer usa .recognize() (no .detect())
        # Resultado tiene los mismos campos que HandLandmarkerResult:
        #   resultado.hand_landmarks[i]   → landmarks de la mano i
        #   resultado.handedness[i]       → lateralidad de la mano i
        # MÁS el campo nuevo:
        #   resultado.gestures[i][0].category_name  → nombre del gesto (ML)
        #   resultado.gestures[i][0].score          → confianza del gesto

        if not resultado.hand_landmarks:
            print("⚠️  No se detectaron manos.")
            return imagen_anotada

        print("=" * 55)
        print("  🤚 ANÁLISIS DE GESTOS (GestureRecognizer)")
        print("=" * 55)

        for idx in range(len(resultado.hand_landmarks)):
            hand_lms   = resultado.hand_landmarks[idx]
            handedness = resultado.handedness[idx]
            etiqueta   = handedness[0].category_name

            # ── Gesto reconocido por ML ────────────────────────────
            gesto_ml       = resultado.gestures[idx][0].category_name
            confianza_gesto = resultado.gestures[idx][0].score

            # ── Gesto por lógica geométrica (fallback / comparación) ─
            dedos           = detectar_dedos_extendidos(hand_lms)
            emoji_geo, nombre_geo = clasificar_gesto(dedos)

            print(f"  Mano {etiqueta}:")
            print(f"    🤖 Gesto ML       : {gesto_ml} ({confianza_gesto:.1%})")
            print(f"    📐 Gesto geométrico: {emoji_geo} {nombre_geo}")
            print(f"    Dedos: ", end='')
            for dedo, ext in dedos.items():
                print(f"{dedo[:3]}{'↑' if ext else '↓'}", end=' ')
            print()

            # Dibujar esqueleto
            mp_drawing.draw_landmarks(
                imagen_anotada, hand_lms,
                HandLandmarksConnections.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style(),
            )

            # Mostrar gesto ML en la imagen
            muneca = hand_lms[0]
            px_m = int(muneca.x * ancho); py_m = int(muneca.y * alto)
            texto_gesto = f"{gesto_ml} ({confianza_gesto:.0%})"
            (tw, th), _ = cv2.getTextSize(texto_gesto, cv2.FONT_HERSHEY_DUPLEX, 0.7, 2)
            cv2.rectangle(imagen_anotada, (px_m-5, py_m+10), (px_m+tw+10, py_m+th+25), (0,0,0), -1)
            cv2.putText(imagen_anotada, texto_gesto, (px_m, py_m+th+15),
                       cv2.FONT_HERSHEY_DUPLEX, 0.7, (0, 255, 200), 2)

        print("=" * 55)

    return imagen_anotada


# Widget
uploader_gesto = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='✊ Subir foto de gesto')
btn_gesto = widgets.Button(description='Detectar Gesto', button_style='info')
salida_gesto = widgets.Output()
display(widgets.VBox([widgets.HTML('<b>Sube una foto con un gesto de mano:</b>'),
                      uploader_gesto, btn_gesto, salida_gesto]))

def on_gesto(_):
    with salida_gesto:
        clear_output()
        if not uploader_gesto.value: print("⚠️  Primero sube una imagen."); return
        valor = uploader_gesto.value
        archivo = valor[0] if isinstance(valor,(list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo,dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada = analizar_gestos_imagen(img_rgb)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Gesto detectado')

btn_gesto.on_click(on_gesto)
print("✅ analizar_gestos_imagen() definida con GestureRecognizer (API mp.tasks).")
print()
print("📌 Mejora vs API legacy:")
print("   Antes: lógica geométrica manual de dedos")
print("   Ahora: GestureRecognizer ML + lógica geométrica como respaldo")
print("   Nuevo campo: resultado.gestures[i][0].category_name")


✅ analizar_gestos_imagen() definida con GestureRecognizer (API mp.tasks).

📌 Mejora vs API legacy:
   Antes: lógica geométrica manual de dedos
   Ahora: GestureRecognizer ML + lógica geométrica como respaldo
   Nuevo campo: resultado.gestures[i][0].category_name


---
# 🚨 MÓDULO 6 — Detección de Caídas y Posturas Incorrectas

## Fundamento técnico

Las caídas y posturas incorrectas se detectan analizando las **relaciones geométricas**  
entre los landmarks del cuerpo:

```
CAÍDA:
  ▸ La cadera baja a nivel de los pies (y_cadera ≈ y_tobillo)
  ▸ El ángulo del tronco es casi horizontal (< 30°)
  ▸ Los hombros están muy cerca del suelo

ESPALDA ENCORVADA:
  ▸ El ángulo hombro-cadera-rodilla se desvía del rango normal (170°-180°)
  ▸ La nariz está desplazada horizontalmente respecto a los hombros

CABEZA INCLINADA:
  ▸ La nariz baja por debajo del nivel de los hombros
  ▸ El ángulo nariz-hombro es mayor a 30°
```

In [15]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 6 — DETECCIÓN DE CAÍDAS Y POSTURAS INCORRECTAS         ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# El algoritmo de detección no cambia.
# Solo cambia cómo se crea el detector y se accede a los landmarks.

UMBRALES_POSTURA = {
    'caida_diferencia_y':  0.15,
    'caida_angulo_tronco': 45,
    'espalda_angulo_min':  150,
    'cabeza_inclinacion':  25,
    'hombros_nivel_max':   15,
}


def calcular_angulo_tronco(landmarks_np):
    """Sin cambios — recibe dict {nombre: [x,y]} con coordenadas normalizadas."""
    hombro_izq = np.array(landmarks_np['hombro_izq'])
    hombro_der = np.array(landmarks_np['hombro_der'])
    centro_hombros = (hombro_izq + hombro_der) / 2

    cadera_izq = np.array(landmarks_np['cadera_izq'])
    cadera_der = np.array(landmarks_np['cadera_der'])
    centro_caderas = (cadera_izq + cadera_der) / 2

    vector_tronco = centro_hombros - centro_caderas
    vertical = np.array([0, -1])
    coseno = np.dot(vector_tronco, vertical) / (
        np.linalg.norm(vector_tronco) * np.linalg.norm(vertical) + 1e-6
    )
    return round(math.degrees(math.acos(np.clip(coseno, -1, 1))), 1)


def detectar_anomalias_postura(imagen_rgb):
    """
    Analiza la postura y detecta caídas, espalda encorvada y asimetría de hombros.
    
    Cambio respecto a la API legacy:
      mp_pose.Pose(...)               →  PoseLandmarkerOptions(...)
      resultado.pose_landmarks.landmark  →  resultado.pose_landmarks[0]
    """
    imagen_anotada = imagen_rgb.copy()
    alto, ancho    = imagen_rgb.shape[:2]

    opts = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_POSE_FULL),
        running_mode=RunningMode.IMAGE, num_poses=1,
        min_pose_detection_confidence=0.5,
    )

    with PoseLandmarker.create_from_options(opts) as detector:
        resultado = detector.detect(rgb_a_mp_image(imagen_rgb))

        if not resultado.pose_landmarks:
            print("⚠️  No se detectó persona.")
            return imagen_anotada, []

        # Nueva API: resultado.pose_landmarks[0] = lista plana de 33 landmarks
        lms = resultado.pose_landmarks[0]

        # Construir dict de coordenadas + visibilidad
        coords = {}
        for nombre, idx in POSE_LANDMARKS.items():
            lm = lms[idx]
            coords[nombre] = [lm.x, lm.y, lm.visibility]

        # Dibujar esqueleto
        mp_drawing.draw_landmarks(
            imagen_anotada, lms,
            PoseLandmarksConnections.POSE_LANDMARKS,
            mp_drawing_styles.get_default_pose_landmarks_style(),
        )

        alertas = []

        # DETECCIÓN 1: CAÍDA
        if coords['cadera_izq'][2] > 0.5 and coords['tobillo_izq'][2] > 0.5:
            diff_y = coords['tobillo_izq'][1] - coords['cadera_izq'][1]
            angulo_tronco = calcular_angulo_tronco({k: v[:2] for k, v in coords.items()})

            if (diff_y < UMBRALES_POSTURA['caida_diferencia_y'] or
                    angulo_tronco > UMBRALES_POSTURA['caida_angulo_tronco']):
                alertas.append(('🚨 CAÍDA DETECTADA', (0, 0, 255)))
                cv2.rectangle(imagen_anotada, (5, 5), (ancho-5, alto-5), (0, 0, 255), 8)

        # DETECCIÓN 2: ESPALDA ENCORVADA
        if all(coords[p][2] > 0.5 for p in ['hombro_izq', 'cadera_izq', 'rodilla_izq']):
            angulo_espalda = calcular_angulo(
                coords['hombro_izq'][:2], coords['cadera_izq'][:2], coords['rodilla_izq'][:2]
            )
            if angulo_espalda < UMBRALES_POSTURA['espalda_angulo_min']:
                alertas.append((f'⚠️  ESPALDA ENCORVADA ({angulo_espalda}°)', (0, 165, 255)))

        # DETECCIÓN 3: ASIMETRÍA DE HOMBROS
        if coords['hombro_izq'][2] > 0.5 and coords['hombro_der'][2] > 0.5:
            diff_hombros = abs(coords['hombro_izq'][1] - coords['hombro_der'][1]) * alto
            if diff_hombros > UMBRALES_POSTURA['hombros_nivel_max']:
                alertas.append((f'⚠️  HOMBROS ASIMÉTRICOS ({diff_hombros:.0f}px dif)', (0, 200, 200)))

        if not alertas:
            alertas.append(('✅ POSTURA CORRECTA', (0, 200, 0)))

        for i, (alerta, color) in enumerate(alertas):
            y_pos = 50 + i * 45
            (tw, th), _ = cv2.getTextSize(alerta, cv2.FONT_HERSHEY_DUPLEX, 0.9, 2)
            cv2.rectangle(imagen_anotada, (10, y_pos-30), (tw+20, y_pos+10), (0,0,0), -1)
            cv2.putText(imagen_anotada, alerta, (15, y_pos), cv2.FONT_HERSHEY_DUPLEX, 0.9, color, 2)

        print("=" * 55); print("  📊 REPORTE DE POSTURA"); print("=" * 55)
        for alerta, _ in alertas: print(f"  {alerta}")
        print("=" * 55)

    return imagen_anotada, alertas


# Widget
uploader_postura = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='🧍 Analizar postura')
btn_postura = widgets.Button(description='Detectar Anomalías', button_style='danger')
salida_postura = widgets.Output()
display(widgets.VBox([widgets.HTML('<b>Sube una imagen de cuerpo completo para análisis de postura:</b>'),
                      uploader_postura, btn_postura, salida_postura]))

def on_postura(_):
    with salida_postura:
        clear_output()
        if not uploader_postura.value: print("⚠️  Primero sube una imagen."); return
        valor = uploader_postura.value
        archivo = valor[0] if isinstance(valor,(list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo,dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        img_anotada, alertas = detectar_anomalias_postura(img_rgb)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'Análisis de postura')

btn_postura.on_click(on_postura)
print("✅ detectar_anomalias_postura() definida.")


✅ detectar_anomalias_postura() definida.


---
# 🎥 MÓDULO 7 — Aplicación en Tiempo Real con Webcam

## Integración completa

Esta celda combina **todos los módulos anteriores** en una sola aplicación  
que funciona en tiempo real con tu webcam.

### Funciones activadas simultáneamente:
- ✋ Detección de gestos de mano
- 🧍 Landmarks del cuerpo + ángulos
- 🚨 Alerta de caídas
- 📊 Contador de FPS en tiempo real

### Controles:
- **▶ Iniciar** → activa la webcam
- **⏹ Detener** → cierra la cámara
- **📸 Capturar** → guarda el fotograma actual

In [17]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 7 — APLICACIÓN EN TIEMPO REAL (WEBCAM)                 ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# CAMBIOS EN TIEMPO REAL (RunningMode.VIDEO):
# ────────────────────────────────────────────
# Para video (múltiples frames con timestamps) la nueva API ofrece:
#   RunningMode.IMAGE  → cada frame es independiente (más simple, algo más lento)
#   RunningMode.VIDEO  → el detector usa tracking entre frames (más eficiente)
#
# Para simplicidad pedagógica, usamos RunningMode.IMAGE en el bucle.
# Los detectores se crean UNA VEZ fuera del bucle (igual que la API legacy).
#
# Diferencia en el modo VIDEO:
#   En lugar de: detector.detect(mp_imagen)
#   Se usa:      detector.detect_for_video(mp_imagen, timestamp_ms)
#
# NOTA: RunningMode.LIVE_STREAM usa callbacks asíncronos y es más complejo.
# Para Colab + webcam, RunningMode.IMAGE es la opción más compatible.

estado_rt = {
    'activo':    False,
    'capturar':  False,
    'modo':      'pose',
    'fps':       0,
    'frame_cap': None,
}


def frame_a_jpeg(frame_bgr):
    """Convierte frame BGR a bytes JPEG para widgets.Image."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    buf = io.BytesIO(); Image.fromarray(rgb).save(buf, format='JPEG', quality=75)
    return buf.getvalue()


def procesar_frame_pose(frame_rgb, pose_detector):
    """
    Detecta pose en un frame de video usando la nueva API.
    
    Cambio respecto a la API legacy:
      pose_processor.process(frame_rgb)     →  pose_detector.detect(mp.Image(...))
      res.pose_landmarks.landmark[i]        →  res.pose_landmarks[0][i]
    """
    frame_anotado = frame_rgb.copy()
    alto, ancho   = frame_rgb.shape[:2]

    resultado = pose_detector.detect(rgb_a_mp_image(frame_rgb))

    if resultado.pose_landmarks:
        lms = resultado.pose_landmarks[0]

        mp_drawing.draw_landmarks(
            frame_anotado, lms,
            PoseLandmarksConnections.POSE_LANDMARKS,
            mp_drawing_styles.get_default_pose_landmarks_style(),
            mp_drawing.DrawingSpec(color=(255, 200, 0), thickness=2),
        )

        # Ángulos en tiempo real (solo los más importantes)
        angulos_rt = [
            ('Codo.I',  11, 13, 15),
            ('Codo.D',  12, 14, 16),
            ('Rod.I',   23, 25, 27),
            ('Rod.D',   24, 26, 28),
        ]
        for nombre, ia, ib, ic in angulos_rt:
            la, lb, lc = lms[ia], lms[ib], lms[ic]
            if la.visibility > 0.5 and lb.visibility > 0.5 and lc.visibility > 0.5:
                ang = calcular_angulo([la.x, la.y], [lb.x, lb.y], [lc.x, lc.y])
                px, py = landmark_a_pixeles(lb, alto, ancho)
                cv2.putText(frame_anotado, f"{ang:.0f}°",
                           (px, py-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,200), 2)

        # Detección de caída simplificada
        cadera_y  = (lms[23].y + lms[24].y) / 2
        tobillo_y = (lms[27].y + lms[28].y) / 2
        if tobillo_y - cadera_y < 0.10:
            cv2.putText(frame_anotado, '🚨 CAIDA', (10, 60),
                       cv2.FONT_HERSHEY_DUPLEX, 1.5, (0, 0, 255), 3)
            cv2.rectangle(frame_anotado, (0, 0), (ancho-1, alto-1), (0, 0, 255), 6)

    return frame_anotado


def iniciar_camara_realtime(indice=0):
    """
    Abre la webcam y ejecuta análisis de pose en tiempo real.
    
    Arquitectura:
      Hilo principal: widgets e interfaz de usuario
      Hilo secundario: captura + procesamiento de frames
      Comunicación: dict 'estado_rt' compartido
    """
    global estado_rt
    estado_rt['activo'] = True

    img_widget   = widgets.Image(format='jpeg', width=640, height=480)
    btn_detener  = widgets.Button(description='⏹ Detener',   button_style='danger')
    btn_capturar = widgets.Button(description='📸 Capturar', button_style='info')
    lbl_info     = widgets.Label(value='Iniciando...')
    salida_cap   = widgets.Output()
    selector_modo = widgets.ToggleButtons(
        options=[('🧍 Pose','pose'), ('✋ Manos','manos'), ('🔮 Completo','holistic')],
        value='pose', description='Modo:',
    )

    def on_detener(b):      estado_rt['activo']  = False
    def on_capturar(b):     estado_rt['capturar']= True
    def on_modo(cambio):    estado_rt['modo']    = cambio['new']

    btn_detener.on_click(on_detener)
    btn_capturar.on_click(on_capturar)
    selector_modo.observe(on_modo, names='value')

    display(widgets.VBox([selector_modo,
                          widgets.HBox([btn_detener, btn_capturar, lbl_info]),
                          img_widget, salida_cap]))

    def loop():
        cam = cv2.VideoCapture(indice, cv2.CAP_DSHOW)
        if not cam.isOpened():
            lbl_info.value = f'❌ No se encontró cámara (índice {indice})'; return

        # Crear detectores UNA VEZ — más eficiente que crear/destruir por frame
        # Usamos mode=IMAGE para simplicidad; en producción usar VIDEO+timestamp
        pose_det = PoseLandmarker.create_from_options(PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=MODEL_POSE_LITE),
            running_mode=RunningMode.IMAGE,
            num_poses=1, min_pose_detection_confidence=0.5,
        ))
        hand_det = HandLandmarker.create_from_options(HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=MODEL_HAND),
            running_mode=RunningMode.IMAGE,
            num_hands=2, min_hand_detection_confidence=0.5,
        ))

        tiempo_prev = time.time(); frame_count = 0

        while estado_rt['activo']:
            ret, frame_bgr = cam.read()
            if not ret: break

            frame_bgr = cv2.flip(frame_bgr, 1)   # efecto espejo
            frame_rgb = bgr_a_rgb(frame_bgr)
            mp_img    = rgb_a_mp_image(frame_rgb) # convertir UNA vez por frame

            modo = estado_rt['modo']

            if modo == 'pose':
                frame_anotado = procesar_frame_pose(frame_rgb, pose_det)

            elif modo == 'manos':
                res = hand_det.detect(mp_img)
                frame_anotado = frame_rgb.copy()
                if res.hand_landmarks:
                    for hand_lms in res.hand_landmarks:
                        mp_drawing.draw_landmarks(
                            frame_anotado, hand_lms,
                            HandLandmarksConnections.HAND_CONNECTIONS,
                            mp_drawing_styles.get_default_hand_landmarks_style(),
                            mp_drawing_styles.get_default_hand_connections_style(),
                        )
                        dedos = detectar_dedos_extendidos(hand_lms)
                        emoji_g, nombre_g = clasificar_gesto(dedos)
                        cv2.putText(frame_anotado, nombre_g,
                                   (10, 50), cv2.FONT_HERSHEY_DUPLEX, 1, (0,255,200), 2)

            elif modo == 'holistic':
                # Sin Holistic en la nueva API: combinamos pose + manos
                res_p = pose_det.detect(mp_img)
                res_h = hand_det.detect(mp_img)
                frame_anotado = frame_rgb.copy()
                if res_p.pose_landmarks:
                    mp_drawing.draw_landmarks(
                        frame_anotado, res_p.pose_landmarks[0],
                        PoseLandmarksConnections.POSE_LANDMARKS,
                        mp_drawing_styles.get_default_pose_landmarks_style(),
                    )
                if res_h.hand_landmarks:
                    for hand_lms in res_h.hand_landmarks:
                        mp_drawing.draw_landmarks(
                            frame_anotado, hand_lms,
                            HandLandmarksConnections.HAND_CONNECTIONS,
                        )
            else:
                frame_anotado = frame_rgb.copy()

            # Calcular y mostrar FPS
            frame_count += 1
            if frame_count % 10 == 0:
                t_actual = time.time()
                fps = 10 / (t_actual - tiempo_prev + 1e-6)
                estado_rt['fps'] = round(fps, 1)
                tiempo_prev = t_actual
                lbl_info.value = f'FPS: {estado_rt["fps"]} | Modo: {modo}'

            cv2.putText(frame_anotado, f"FPS: {estado_rt['fps']}",
                       (10, frame_anotado.shape[0]-15),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

            img_widget.value = frame_a_jpeg(cv2.cvtColor(frame_anotado, cv2.COLOR_RGB2BGR))

            if estado_rt['capturar']:
                estado_rt['capturar'] = False
                estado_rt['frame_cap'] = frame_anotado.copy()
                with salida_cap:
                    clear_output(); print("📸 Frame capturado:")
                    mostrar_imagen(frame_anotado, 'Captura', figsize=(8,5))

            time.sleep(0.01)

        cam.release(); pose_det.close(); hand_det.close()
        lbl_info.value = '✅ Cámara cerrada.'

    threading.Thread(target=loop, daemon=True).start()


print("📹 Módulo 7 listo.")
print()
print("⚙️  Controles: 🧍 Pose | ✋ Manos | 🔮 Completo")
print()
print("📌 Cambios en tiempo real:")
print("   pose_processor.process(frame)     →  pose_detector.detect(mp.Image(...))")
print("   result.pose_landmarks.landmark[i] →  result.pose_landmarks[0][i]")
print()
iniciar_camara_realtime(indice=1)


📹 Módulo 7 listo.

⚙️  Controles: 🧍 Pose | ✋ Manos | 🔮 Completo

📌 Cambios en tiempo real:
   pose_processor.process(frame)     →  pose_detector.detect(mp.Image(...))
   result.pose_landmarks.landmark[i] →  result.pose_landmarks[0][i]



---
# 🟡 MÓDULO 8 — YOLOv8-pose como Tecnología Alternativa

## ¿Qué es YOLOv8-pose?

YOLO (You Only Look Once) es una familia de modelos de detección de objetos.  
La versión **YOLOv8-pose** extiende YOLO para detectar simultáneamente:
- 📦 El **bounding box** de cada persona (rectángulo delimitador)
- 🦾 Los **17 keypoints** COCO del cuerpo (estándar diferente a MediaPipe)

## Comparativa: MediaPipe vs YOLOv8-pose

| Aspecto | MediaPipe Pose | YOLOv8-pose |
|---|---|---|
| Landmarks | 33 | 17 (estándar COCO) |
| Multi-persona | No (1 por defecto) | ✅ Sí, múltiples |
| Velocidad | ⚡ Muy rápido | ⚡ Rápido |
| Face/Hands | ✅ Sí (con Holistic) | ❌ No |
| Bounding Box | ❌ No | ✅ Sí |

## Keypoints COCO (17 puntos)
```
0=nariz, 1=ojo_izq, 2=ojo_der, 3=oreja_izq, 4=oreja_der
5=hombro_izq, 6=hombro_der, 7=codo_izq, 8=codo_der
9=muneca_izq, 10=muneca_der, 11=cadera_izq, 12=cadera_der
13=rodilla_izq, 14=rodilla_der, 15=tobillo_izq, 16=tobillo_der
```

In [18]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 8 — YOLOv8-POSE                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

from ultralytics import YOLO

# Nombres de los 17 keypoints del estándar COCO
COCO_KEYPOINTS = [
    'nariz',       # 0
    'ojo_izq',     # 1
    'ojo_der',     # 2
    'oreja_izq',   # 3
    'oreja_der',   # 4
    'hombro_izq',  # 5
    'hombro_der',  # 6
    'codo_izq',    # 7
    'codo_der',    # 8
    'muneca_izq',  # 9
    'muneca_der',  # 10
    'cadera_izq',  # 11
    'cadera_der',  # 12
    'rodilla_izq', # 13
    'rodilla_der', # 14
    'tobillo_izq', # 15
    'tobillo_der', # 16
]

# Conexiones para dibujar el esqueleto (pares de índices)
COCO_SKELETON = [
    (0, 1), (0, 2),               # nariz-ojos
    (1, 3), (2, 4),               # ojos-orejas
    (5, 6),                       # hombros entre sí
    (5, 7), (7, 9),               # brazo izquierdo
    (6, 8), (8, 10),              # brazo derecho
    (5, 11), (6, 12),             # torso
    (11, 12),                     # caderas entre sí
    (11, 13), (13, 15),           # pierna izquierda
    (12, 14), (14, 16),           # pierna derecha
]


def analizar_con_yolopose(imagen_rgb, conf_umbral=0.5):
    """
    Detecta personas y sus keypoints usando YOLOv8-pose.
    Soporta detección de MÚLTIPLES personas simultáneamente.
    
    Parámetros:
        imagen_rgb    → array NumPy RGB
        conf_umbral   → confianza mínima para considerar una detección (0.0-1.0)
    """
    imagen_anotada = imagen_rgb.copy()

    # ── Cargar el modelo YOLOv8-pose ──────────────────────────────
    # 'yolov8n-pose.pt' = variante 'nano' (n) → más pequeño y rápido
    # Opciones: yolov8n-pose, yolov8s-pose, yolov8m-pose, yolov8l-pose, yolov8x-pose
    #           nano < small < medium < large < extra-large
    # Si no está descargado, se descarga automáticamente (~6MB para nano)

    print("⬇️  Cargando modelo YOLOv8n-pose...")
    modelo = YOLO('yolov8n-pose.pt')
    print("✅ Modelo listo.")

    # ── Ejecutar inferencia ───────────────────────────────────────
    # model.predict() devuelve una lista de resultados (uno por imagen).
    # verbose=False suprime la barra de progreso en consola.
    resultados = modelo.predict(
        source  = imagen_rgb,   # puede ser array NumPy, ruta de archivo, URL, video...
        conf    = conf_umbral,  # umbral de confianza mínima para detecciones
        verbose = False
    )

    res = resultados[0]  # tomamos el primer (y único) resultado de la lista

    # ── Verificar detecciones ─────────────────────────────────────
    # res.keypoints contiene los keypoints detectados.
    # .data tiene forma (N_personas, 17, 3) donde 3 = [x, y, confianza]

    if res.keypoints is None or len(res.keypoints.data) == 0:
        print("⚠️  No se detectaron personas.")
        return imagen_anotada, pd.DataFrame()

    n_personas = len(res.keypoints.data)
    print(f"✅ {n_personas} persona(s) detectada(s).")

    # Colores para cada persona (hasta 5 colores diferentes)
    COLORES_PERSONAS = [
        (255, 100,   0),   # naranja
        (  0, 200, 100),   # verde
        (100,   0, 255),   # morado
        (255,   0, 150),   # rosa
        (  0, 150, 255),   # azul
    ]

    todas_filas = []

    # ── Procesar cada persona detectada ───────────────────────────
    for idx_persona in range(n_personas):
        color = COLORES_PERSONAS[idx_persona % len(COLORES_PERSONAS)]

        # ── Dibujar bounding box ───────────────────────────────────
        # res.boxes.xyxy → tensor con [x1, y1, x2, y2] de cada bounding box
        if res.boxes is not None and idx_persona < len(res.boxes):
            box = res.boxes.xyxy[idx_persona].cpu().numpy().astype(int)
            # .cpu() mueve el tensor de GPU a CPU
            # .numpy() convierte a array NumPy
            # .astype(int) convierte a enteros para cv2
            cv2.rectangle(imagen_anotada,
                         (box[0], box[1]), (box[2], box[3]),
                         color, 2)
            confianza_box = float(res.boxes.conf[idx_persona])
            cv2.putText(imagen_anotada,
                       f"Persona {idx_persona+1} ({confianza_box:.0%})",
                       (box[0], box[1]-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # ── Extraer keypoints ──────────────────────────────────────
        # keypoints.data[i] → tensor de forma (17, 3): [x, y, confianza] por punto
        kpts = res.keypoints.data[idx_persona].cpu().numpy()

        # ── Dibujar conexiones del esqueleto ──────────────────────
        for (i1, i2) in COCO_SKELETON:
            x1, y1, c1 = kpts[i1]
            x2, y2, c2 = kpts[i2]
            # Solo dibujamos si ambos puntos tienen confianza > 0.5
            if c1 > 0.5 and c2 > 0.5:
                cv2.line(imagen_anotada,
                        (int(x1), int(y1)), (int(x2), int(y2)),
                        color, 2)

        # ── Dibujar puntos y etiquetas ────────────────────────────
        for idx_kpt, (x, y, conf) in enumerate(kpts):
            if conf > 0.5:
                px, py = int(x), int(y)
                cv2.circle(imagen_anotada, (px, py), 5, color, -1)
                cv2.circle(imagen_anotada, (px, py), 5, (255,255,255), 1)  # borde blanco
                nombre = COCO_KEYPOINTS[idx_kpt]
                cv2.putText(imagen_anotada, nombre[:5],
                           (px+6, py-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 200), 1)

            # Guardar en tabla
            todas_filas.append({
                'Persona':       idx_persona + 1,
                'Keypoint':      COCO_KEYPOINTS[idx_kpt],
                'Índice COCO':   idx_kpt,
                'x (píxeles)':  round(float(x), 1),
                'y (píxeles)':  round(float(y), 1),
                'Confianza':     round(float(conf), 3),
            })

    tabla = pd.DataFrame(todas_filas)
    return imagen_anotada, tabla


# Widget
uploader_yolo = widgets.FileUpload(
    accept='.jpg,.jpeg,.png', multiple=False, description='🟡 Subir imagen (YOLO)'
)
btn_yolo = widgets.Button(description='Analizar con YOLO', button_style='warning')
salida_yolo = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen (puede contener múltiples personas):</b>'),
    uploader_yolo, btn_yolo, salida_yolo
]))

def on_yolo(_):
    with salida_yolo:
        clear_output()
        if not uploader_yolo.value:
            print("⚠️  Primero sube una imagen."); return
        valor = uploader_yolo.value
        archivo = valor[0] if isinstance(valor,(list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo,dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))

        img_anotada, tabla = analizar_con_yolopose(img_rgb)
        mostrar_comparacion(img_rgb, img_anotada, 'Original', 'YOLOv8-pose (17 keypoints COCO)')

        if not tabla.empty:
            print("\n📊 Keypoints detectados:")
            display(tabla[tabla['Confianza'] > 0.5])

btn_yolo.on_click(on_yolo)

---
# 📊 MÓDULO 9 — Comparativa de Tecnologías

## ¿Cuándo usar cada tecnología?

Esta celda presenta un cuadro resumen y ejecuta una **comparación de velocidad**  
entre MediaPipe Lite, MediaPipe Full y YOLOv8n para que puedas elegir  
la mejor opción según tu caso de uso.

In [19]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 9 — COMPARATIVA DE TECNOLOGÍAS DE POSE                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Tabla comparativa cualitativa ─────────────────────────────────
comparativa = pd.DataFrame([
    {
        'Tecnología':        'MediaPipe Pose (lite)',
        'Landmarks':         '33 (cuerpo)',
        'Multi-persona':     'No',
        'Face+Hands':        'Solo con Holistic',
        'Velocidad':         '⚡⚡⚡ Muy alta',
        'Precisión':         '⭐⭐⭐',
        'Mejor para':        'Tiempo real móvil/PC',
        'Offline':           '✅ Sí',
    },
    {
        'Tecnología':        'MediaPipe Pose (full)',
        'Landmarks':         '33 (cuerpo)',
        'Multi-persona':     'No',
        'Face+Hands':        'Solo con Holistic',
        'Velocidad':         '⚡⚡ Alta',
        'Precisión':         '⭐⭐⭐⭐',
        'Mejor para':        'Análisis biomecánico',
        'Offline':           '✅ Sí',
    },
    {
        'Tecnología':        'MediaPipe Holistic',
        'Landmarks':         '543 (todo)',
        'Multi-persona':     'No',
        'Face+Hands':        '✅ Incluido',
        'Velocidad':         '⚡ Moderada',
        'Precisión':         '⭐⭐⭐⭐⭐',
        'Mejor para':        'Lenguaje señas, AR',
        'Offline':           '✅ Sí',
    },
    {
        'Tecnología':        'YOLOv8n-pose',
        'Landmarks':         '17 COCO (cuerpo)',
        'Multi-persona':     '✅ Sí',
        'Face+Hands':        'No',
        'Velocidad':         '⚡⚡⚡ Muy alta',
        'Precisión':         '⭐⭐⭐⭐',
        'Mejor para':        'Multi-persona, seguridad',
        'Offline':           '✅ Sí',
    },
    {
        'Tecnología':        'YOLOv8x-pose',
        'Landmarks':         '17 COCO (cuerpo)',
        'Multi-persona':     '✅ Sí',
        'Face+Hands':        'No',
        'Velocidad':         '⚡ Baja (GPU recomendada)',
        'Precisión':         '⭐⭐⭐⭐⭐',
        'Mejor para':        'Análisis offline preciso',
        'Offline':           '✅ Sí',
    },
    {
        'Tecnología':        'OpenPose',
        'Landmarks':         '25 (cuerpo) + cara/manos',
        'Multi-persona':     '✅ Sí',
        'Face+Hands':        '✅ Opcional',
        'Velocidad':         'Baja (requiere GPU)',
        'Precisión':         '⭐⭐⭐⭐⭐',
        'Mejor para':        'Investigación, datasets',
        'Offline':           '✅ Sí (complejo instalar)',
    },
])

print("📊 TABLA COMPARATIVA DE TECNOLOGÍAS DE POSE ESTIMATION")
print()
display(comparativa)

# ── Benchmark de velocidad (necesita imagen disponible) ───────────
print()
print("⏱️  BENCHMARK DE VELOCIDAD")
print("   Para medir tiempos reales, ejecuta la siguiente celda")
print("   con una imagen de tu elección.")

📊 TABLA COMPARATIVA DE TECNOLOGÍAS DE POSE ESTIMATION



,Tecnología,Landmarks,Multi-persona,Face+Hands,Velocidad,Precisión,Mejor para,Offline
0,MediaPipe Pose (lite),33 (cuerpo),No,Solo con Holistic,⚡⚡⚡ Muy alta,⭐⭐⭐,Tiempo real móvil/PC,✅ Sí
1,MediaPipe Pose (full),33 (cuerpo),No,Solo con Holistic,⚡⚡ Alta,⭐⭐⭐⭐,Análisis biomecánico,✅ Sí
2,MediaPipe Holistic,543 (todo),No,✅ Incluido,⚡ Moderada,⭐⭐⭐⭐⭐,"Lenguaje señas, AR",✅ Sí
3,YOLOv8n-pose,17 COCO (cuerpo),✅ Sí,No,⚡⚡⚡ Muy alta,⭐⭐⭐⭐,"Multi-persona, seguridad",✅ Sí
4,YOLOv8x-pose,17 COCO (cuerpo),✅ Sí,No,⚡ Baja (GPU recomendada),⭐⭐⭐⭐⭐,Análisis offline preciso,✅ Sí
5,OpenPose,25 (cuerpo) + cara/manos,✅ Sí,✅ Opcional,Baja (requiere GPU),⭐⭐⭐⭐⭐,"Investigación, datasets",✅ Sí (complejo instalar)



⏱️  BENCHMARK DE VELOCIDAD
   Para medir tiempos reales, ejecuta la siguiente celda
   con una imagen de tu elección.


In [20]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  MÓDULO 9.B — BENCHMARK COMPARATIVO (sube tu imagen primero)   ║
# ╚══════════════════════════════════════════════════════════════════╝

uploader_bench = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='📏 Imagen para benchmark')
btn_bench = widgets.Button(description='Ejecutar Benchmark', button_style='warning')
salida_bench = widgets.Output()
display(widgets.VBox([uploader_bench, btn_bench, salida_bench]))


def on_benchmark(_):
    with salida_bench:
        clear_output()
        if not uploader_bench.value: print("⚠️  Sube una imagen primero."); return
        valor = uploader_bench.value
        archivo = valor[0] if isinstance(valor,(list,tuple)) else list(valor.values())[0]
        contenido = archivo['content'] if isinstance(archivo,dict) else archivo.content
        img_rgb = np.array(Image.open(io.BytesIO(contenido)).convert('RGB'))
        mp_img  = rgb_a_mp_image(img_rgb)   # convertir una vez para todos los modelos

        N = 10
        print(f"🔄 Ejecutando {N} iteraciones por modelo...")
        print()
        resultados_bench = []

        # ── MediaPipe Pose Lite (nueva API) ────────────────────────
        with PoseLandmarker.create_from_options(PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=MODEL_POSE_LITE),
            running_mode=RunningMode.IMAGE,
        )) as det:
            tiempos = []
            for _ in range(N):
                t0 = time.perf_counter(); det.detect(mp_img)
                tiempos.append(time.perf_counter() - t0)
            resultados_bench.append(('MP Pose Lite (mp.tasks)',   np.mean(tiempos)*1000))

        # ── MediaPipe Pose Full (nueva API) ────────────────────────
        with PoseLandmarker.create_from_options(PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=MODEL_POSE_FULL),
            running_mode=RunningMode.IMAGE,
        )) as det:
            tiempos = []
            for _ in range(N):
                t0 = time.perf_counter(); det.detect(mp_img)
                tiempos.append(time.perf_counter() - t0)
            resultados_bench.append(('MP Pose Full (mp.tasks)',   np.mean(tiempos)*1000))

        # ── MediaPipe Hand Landmarker ──────────────────────────────
        with HandLandmarker.create_from_options(HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=MODEL_HAND),
            running_mode=RunningMode.IMAGE,
        )) as det:
            tiempos = []
            for _ in range(N):
                t0 = time.perf_counter(); det.detect(mp_img)
                tiempos.append(time.perf_counter() - t0)
            resultados_bench.append(('MP Hand Landmarker (mp.tasks)', np.mean(tiempos)*1000))

        # ── YOLOv8n-pose ───────────────────────────────────────────
        try:
            from ultralytics import YOLO
            model_yolo = YOLO('yolov8n-pose.pt')
            img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
            tiempos = []
            for _ in range(N):
                t0 = time.perf_counter()
                model_yolo(img_bgr, verbose=False)
                tiempos.append(time.perf_counter() - t0)
            resultados_bench.append(('YOLOv8n-pose', np.mean(tiempos)*1000))
        except Exception as e:
            print(f"   YOLOv8 no disponible: {e}")

        # ── Mostrar resultados ─────────────────────────────────────
        print(f"{'Modelo':<35} {'ms/frame':>10}  {'FPS est.':>10}")
        print("─" * 60)
        for nombre, ms in resultados_bench:
            fps_est = round(1000/ms, 1)
            print(f"  {nombre:<33} {ms:>9.1f}ms  {fps_est:>8.1f} fps")

        fig, ax = plt.subplots(figsize=(10, 4))
        nombres  = [r[0] for r in resultados_bench]
        tiempos  = [r[1] for r in resultados_bench]
        colores  = ['#2196F3' if 'MP' in n else '#FF5722' for n in nombres]
        ax.barh(nombres, tiempos, color=colores, edgecolor='white')
        for i, (n, t) in enumerate(zip(nombres, tiempos)):
            ax.text(t+1, i, f'{t:.1f}ms', va='center', fontsize=10)
        ax.set_xlabel('Tiempo promedio por frame (ms)'); ax.set_xlim(0, max(tiempos)*1.3)
        ax.set_title('⏱ Benchmark — Nueva API mp.tasks vs YOLOv8', fontsize=13)
        plt.tight_layout(); plt.show()


btn_bench.on_click(on_benchmark)
print("✅ Benchmark listo.")
print("📌 Benchmark actualizado a la nueva API mp.tasks.")


✅ Benchmark listo.
📌 Benchmark actualizado a la nueva API mp.tasks.


---
# 🎓 MÓDULO 10 — Resumen, Ejercicios y Próximos Pasos

## ¿Qué aprendiste en este notebook?

| Módulo | Habilidad adquirida |
|---|---|
| 0 | Configurar el entorno, importar librerías, funciones utilitarias |
| 1 | Face Mesh: 468 landmarks, EAR/MAR, detección de somnolencia |
| 2 | Hand Landmarks: 21 puntos, esqueleto de la mano |
| 3 | Body Pose: 33 puntos, ángulos biomecánicos articulares |
| 4 | Holistic: 543 landmarks simultáneos con una sola llamada |
| 5 | Detección de gestos por lógica de dedos extendidos |
| 6 | Detección de caídas y posturas incorrectas |
| 7 | Aplicación completa en tiempo real con webcam y threading |
| 8 | YOLOv8-pose: 17 keypoints COCO, multi-persona |
| 9 | Benchmark comparativo de tecnologías |

---

## 🚀 Ejercicios propuestos

### Nivel Básico
1. **Mapa de calor de visibilidad**: para un video de 30 segundos, calcula la visibilidad promedio de cada landmark del cuerpo y dibuja un mapa de calor.
2. **Contador de repeticiones**: cuenta cuántas veces el ángulo del codo baja de 90° (flexión) para contar flexiones de bíceps.
3. **Nuevo gesto**: añade un gesto nuevo al módulo 5 (por ejemplo: 🖖 saludo vulcano = índice+medio separados del anular+meñique).

### Nivel Intermedio
4. **Análisis postural de escritorio**: detecta si el usuario está encorvado frente a la computadora y lanza una alerta de audio con `playsound`.
5. **Reconocimiento de letras ASL**: crea un clasificador de las 26 letras del alfabeto de señas americano usando los 63 valores (21 landmarks × 3 coords) de la mano como features.
6. **Tracker de trayectorias**: dibuja la trayectoria de las muñecas durante el video (cola de 30 frames) con un degradado de color para mostrar el movimiento.

### Nivel Avanzado
7. **Análisis de ejercicio completo**: crea una aplicación que analice una sentadilla (squat) y de retroalimentación sobre: ángulo de rodillas, alineación de rodillas con pies, profundidad de la sentadilla.
8. **Multi-persona + identificación**: usa YOLOv8-pose para detectar múltiples personas y asigna un ID de seguimiento persistente entre frames.
9. **Exportar a CSV para ML**: guarda los 33 landmarks + 10 ángulos de cada frame en un CSV. Luego entrena un clasificador Random Forest para reconocer posturas (de pie, sentado, con los brazos levantados, etc.).

---

## 📚 Recursos para seguir aprendiendo

```
📖 Documentación oficial:
   MediaPipe → https://mediapipe.dev
   YOLOv8   → https://docs.ultralytics.com
   OpenCV   → https://docs.opencv.org

📄 Artículos científicos clave:
   BlazePose (MediaPipe) → arxiv.org/abs/2006.10204
   OpenPose             → arxiv.org/abs/1812.08008
   YOLOv8              → github.com/ultralytics/ultralytics

🗃️ Datasets para entrenar tus propios modelos:
   COCO Keypoints  → cocodataset.org
   Human3.6M       → vision.imar.ro/human3.6m
   MPII Human Pose → human-pose.mpi-inf.mpg.de
```

---

## 🏗️ Arquitectura para proyectos en producción

```
ENTRADA
  ├── Cámara IP (RTSP)     → cv2.VideoCapture('rtsp://...')
  ├── Archivo de video     → cv2.VideoCapture('video.mp4')
  ├── Stream en la nube    → cv2.VideoCapture(url)
  └── Batch de imágenes    → bucle sobre lista de archivos

PROCESAMIENTO
  ├── MediaPipe (1 persona, tiempo real)
  ├── YOLOv8-pose (múltiples personas)
  └── Modelos personalizados (TFLite, ONNX)

LÓGICA DE NEGOCIO
  ├── Detección de caídas → alerta SMS/push
  ├── Análisis deportivo  → métricas y feedback
  ├── Control de UI       → gestos = comandos
  └── Exportar datos      → CSV, base de datos

SALIDA
  ├── Video anotado       → archivo .mp4
  ├── Dashboard web       → Flask/Streamlit
  ├── Alertas             → email, SMS, webhook
  └── Datos               → CSV, JSON, DB
```